# Modeling: Swire Coca‑Cola

- Authors: Ali Ladha, Cyrus Sobhani, Robby Stohel, and Sterling LeDuc
- Date: 2025-10-30

# Table of Contents

1 [Modeling Swire Coca-Cola](#Modeling:-Swire-Coca---Cola)
 
2 [Table of Contents](#Table-of-Contents)
 
3 [Introduction](#Introduction)
> 3.1 [Initial Guiding Questions](#Initial-Guiding-Questions)
 
> 3.2 [Team Member Contributions](#Team-Member-Contributions)
 
> 3.3 [Initial Setup](#Initial-Setup)

> 3.4 [Helper Functions](#Helper-Functions)
 
4 [Data Preparation](#Data-Preparation)
> 4.1 [Q1 & Q2 Preparation](#Q1-&-Q2-Preparation)
 
> 4.2 [Q3 & Q4 Preparation](#Q3-&-Q4-Preparation)
 
> 4.3 [Q5 Preparation](#Q5-Preparation)

> 4.4 [Q6 Interpretation](#Q6-Interpretation)
 
5 [Modeling & Performance](#Modeling-&-Performance)
> 5.1 [Q1 & Q2 Modeling](#Q1-&-Q2-Modeling)
 
> 5.2 [Q3 & Q4 Modeling](#Q3-&-Q4-Modeling)
 
> 5.3 [Q5 Modeling](#Q5-Modeling)
 
> 5.4 [Q6 Modeling](#Q6-Modeling)
 
6 [Results](#Results)
> 6.1 [Q1 Behavioral Events & Sequences](#Q1-Behavioral-Events-&-Sequences)
 
> 6.2 [Q2 Behaviors & Conditions](#Q2-Behaviors-&-Conditions)
 
> 6.3 [Q3 Variance by Device Type](#Q3-Variance-by-Device-Type)

> 6.4 [Q4 Product Frequency](#Q4-Product-Frequency)

> 6.5 [Q5 Financial Impact](#Q5-Financial-Impact)

> 6.6 [Q6 Post-Abandonment](#Q6-Post---Abandonment)

> 6.7 [Recommendations](#Recommendations)

# Introduction

Building on the foundation established during the exploratory data analysis (EDA) stage, this modeling phase applies statistical and machine learning techniques to understand and predict cart abandonment behavior within the MyCoke360 digital ordering platform. The MyCoke360 system, launched by Coca-Cola in Summer 2024, supports Food Service On Premise (FSOP) customers such as restaurants, schools, hospitals, and retailers. These customers are vital to Coca-Cola's B2B business, where frequency of orders, product preferences, and customer retention directly influence revenue.

The modeling stage translates insights from the EDA into actionable predictive and explanatory models. It aims to quantify the drivers of cart abandonment, identify the conditions that encourage recovery or re-engagement, and estimate the financial implications of these behaviors. This process follows the modeling step of the CRISP-DM methodology, focusing on developing, testing, and comparing multiple model types to identify the most effective approach for each research question.

Each guiding question is addressed using an appropriate analytical technique. Logistic regression, random forest, and XGBoost are applied to identify behavioral and transactional predictors of abandonment. SARIMA and Prophet models are used to explore temporal and seasonal dynamics in order activity. Clustering (KMeans) and simulation methods help segment customers and forecast potential business outcomes under varying conditions. Text analytics approaches such as n-gram analysis are used to evaluate event sequences and session patterns leading to abandonment or recovery.

Through iterative experimentation and evaluation using methods such as K-Fold cross-validation, model performance is compared to ensure generalizability and practical relevance. The results of this stage will inform actionable recommendations for MyCoke360, including strategies to reduce cart abandonment, improve customer retention, and optimize product and device-level engagement across the platform.

## Initial Guiding Questions

- (Q1) What behavioral events or sequence of events are the strongest predictors of cart abandonment?
- (Q2) What behaviors or conditions lead to a customer returning to complete a previously abandoned cart?
- (Q3) How does cart abandonment vary by device type and how can it be reduced?
- (Q4) Which products appear most frequently in abandoned carts?
- (Q5) What is the financial impact of cart abandonment on total MyCoke360 revenue and product mix?
- (Q6) What happens after a cart is abandoned? Does the customer order through another method or churn?

## Team Member Contributions

Ali engineered the data required for modeling and built multiple models, including Random Forest, Logistic Regression, and XGBoost, along with tuned versions. He contributed model interpretations, visualizations, and completed an individual notebook addressing how cart abandonment varies by device type and which products appear most frequently in abandoned carts.

Cyrus supported the construction of the labeled cart abandonment dataset, engineered data for modeling, and conducted text-based modeling using n-grams and similar methods. His individual notebook focused on identifying behavioral patterns predicting cart abandonment and the factors influencing customer return after abandonment.

Robby engineered modeling data, developed an individual notebook, and consolidated all team notebooks into the final integrated version. He also standardized and cleaned the project codebase. His analysis used multiple modeling and analytical approaches, including initial EDA visuals, SARIMA, Prophet, Logistic Regression, XGBoost, simulation, and clustering (K-Means). His work focused on assessing the financial impact of cart abandonment on overall MyCoke360 revenue and product mix.

Sterling engineered behavioral and churn-related features, analyzed post-abandonment activity, and identified customer channel-switching behavior. He built a Random Forest churn prediction model with feature importance analysis and created visual analytics to interpret engagement patterns. His work showed that abandonment often leads to continued engagement rather than immediate churn. He focused on abandonment rate by device and is leading the presentation design.

## Initial Setup

In [0]:
# Standard library imports
import ast
import json
import math
import warnings
from collections import Counter
from datetime import datetime, timedelta
from itertools import product
from pathlib import Path

# Data handling and visualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
from statsmodels.graphics.gofplots import qqplot
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Time-series modeling
import statsmodels.api as sm
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Machine learning
from joblib import Memory
from scipy.stats import loguniform
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_squared_error,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    TimeSeriesSplit,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

# Gradient boosting
from xgboost import XGBClassifier
# import xgboost as xgb


seed = 24601

# Load core datasets as pandas DataFrames
google_analytics = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_google_analytics_abandonment.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

sales = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_sales.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

materials = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_material.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

customer = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_customer.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

cutoff_times = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_cutoff_times.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

operating_hours = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_operating_hours.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

orders = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_orders.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

visit_plan = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_visit_plan.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

# Standardize column names to lowercase across all dataframes
for df in [
    customer,
    cutoff_times,
    google_analytics,
    materials,
    operating_hours,
    orders,
    sales,
    visit_plan,
]:
    df.columns = df.columns.str.lower()

In [0]:
# Preview loaded data
display(google_analytics.head(3))
display(sales.head(3))
display(materials.head(3))
display(customer.head(3))
display(cutoff_times.head(3))
display(operating_hours.head(3))
display(orders.head(3))
display(visit_plan.head(3))

## Helper Functions

In [0]:
def assign_session_id(df: pd.DataFrame) -> pd.DataFrame:
    """Assign session_id by grouping add_to_cart and isolating purchases."""
    local = df.copy()

    add_mask = local["event_name"] == "add_to_cart"
    buy_mask = local["event_name"] == "purchase"

    add = local[add_mask].copy()
    add["session_id"] = (
        add.groupby(
            [
                "customer_id",
                "device_category",
                "device_operating_system",
                "event_date",
            ]
        ).ngroup()
    )

    buy = local[buy_mask].copy()
    base = int(add["session_id"].max()) + 1 if not add.empty else 0
    buy = buy.reset_index(drop=True)
    buy["session_id"] = buy.index + base

    out = (
        pd.concat([add, buy], axis=0)
        .sort_values("event_ts_utc")
        .reset_index(drop=True)
    )
    out["session_id"] = out["session_id"].astype(int)
    return out


def parse_items(value: object) -> list | None:
    """Parse items field into a list of dicts or return None."""
    if pd.isna(value):
        return None
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        return [value]
    if isinstance(value, str) and value.strip():
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, dict):
                return [parsed]
            if isinstance(parsed, list):
                return parsed
        except Exception:
            return None
    return None


def top_k_bucket(s: pd.Series, k: int) -> pd.Series:
    """Bucket infrequent categories into '__other__'."""
    top = s.value_counts(dropna=True).head(k).index
    return s.where(s.isin(top), "__other__")


def switched(row: pd.Series) -> bool:
    """Return True if after_order_types contains any new types."""
    before = set(row["before_order_types"])
    after = set(row["after_order_types"])
    return not after.issubset(before)


def evaluate_on_test(name: str, fitted_model) -> None:
    """Print classification metrics on the test set."""
    print(f"\n##### {name} — Test Performance #####")
    y_pred = fitted_model.predict(X_test)
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, digits=3))
    if hasattr(fitted_model, "predict_proba"):
        y_prob = fitted_model.predict_proba(X_test)[:, 1]
        ap = average_precision_score(y_test, y_prob)
        print(f"Average Precision (PR-AUC): {ap:.4f}")


def downsample(
    Xin: pd.DataFrame,
    yin: pd.Series,
    ratio: int = 2
) -> tuple[pd.DataFrame, pd.Series]:
    """Downsample majority class to a given ratio."""
    vc = yin.value_counts()
    maj = vc.idxmax()
    mino = vc.idxmin()
    n_min = vc[mino]
    n_maj_tgt = ratio * n_min

    rng = np.random.RandomState(seed)
    maj_idx = yin[yin == maj].index
    keep_maj = rng.choice(
        maj_idx, size=min(n_maj_tgt, len(maj_idx)), replace=False
    )

    keep_idx = np.concatenate([yin[yin == mino].index, keep_maj])
    X_out = Xin.loc[keep_idx].reset_index(drop=True)
    y_out = yin.loc[keep_idx].reset_index(drop=True)
    return X_out, y_out


def build_preprocessor(
    Xtrain: pd.DataFrame
) -> tuple[ColumnTransformer, list[str], list[str]]:
    """Build a numeric+categorical preprocessing pipeline."""
    num_cols = Xtrain.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()
    cat_cols = Xtrain.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    pre = ColumnTransformer(
        [
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ]
    )
    return pre, num_cols, cat_cols


def pred_proba_best(bst, dmat) -> np.ndarray:
    """Predict probabilities using the best iteration/tree limit when set."""
    if hasattr(bst, "best_iteration") and bst.best_iteration is not None:
        try:
            return bst.predict(
                dmat, iteration_range=(0, bst.best_iteration + 1)
            )
        except TypeError:
            pass

    if hasattr(bst, "best_ntree_limit") and bst.best_ntree_limit is not None:
        return bst.predict(dmat, ntree_limit=bst.best_ntree_limit)

    return bst.predict(dmat)


def collapse_runs(events):
    """Collapse consecutive duplicate events."""
    collapsed = []
    previous = object()
    for event in events:
        if event != previous:
            collapsed.append(event)
        previous = event
    return collapsed


def top_counts(series, top_n=15):
    """Get top value counts with percentages."""
    s = series.value_counts(dropna=False)
    total = s.sum()
    s = s.head(top_n)
    return pd.DataFrame(
        {
            "pattern": s.index.to_list(),
            "count": s.values,
            "percent": (s.values / total * 100) if total else [0.0] * len(s),
        }
    )


def stream_ngrams_count(seqs, n):
    """Count n-grams efficiently in sequences."""
    c = Counter()
    for seq in seqs:
        if isinstance(seq, list) and len(seq) >= n:
            # Update directly from generator
            c.update(tuple(seq[i:i + n]) for i in range(len(seq) - n + 1))
    return c


def top_ngrams_df(seqs, n=2, top_n=15):
    """Return DataFrame of top n-grams."""
    c = stream_ngrams_count(seqs, n)
    total = sum(c.values())
    rows = [
        (k, v, (v / total * 100 if total else 0.0))
        for k, v in c.most_common(top_n)
    ]
    return pd.DataFrame(rows, columns=["pattern", "count", "percent"])


def compare_tables(df_a, df_b, label_a="abandoned", label_b="completed"):
    """Compare two summary tables."""
    a = df_a.rename(
        columns={"count": f"count_{label_a}", "percent": f"percent_{label_a}"}
    )
    b = df_b.rename(
        columns={"count": f"count_{label_b}", "percent": f"percent_{label_b}"}
    )

    # Merge small top-N tables
    return a.merge(b, on="pattern", how="outer")


def analyze_avg_time_between(purchvar, category, event_name):
    """Analyze average time between precursor event and purchase."""
    # Clean datatypes
    purch = purchvar[["event_ts_utc", "purchase_segment"]].copy()
    upd = category[["event_ts_utc", "purchase_segment"]].copy()

    purch["event_ts_utc"] = pd.to_datetime(purch["event_ts_utc"], utc=True)
    upd["event_ts_utc"] = pd.to_datetime(upd["event_ts_utc"], utc=True)
    purch["purchase_segment"] = purch["purchase_segment"].astype(str)
    upd["purchase_segment"] = upd["purchase_segment"].astype(str)

    # Merge purchases and precursors
    merged = purch.merge(
        upd,
        on="purchase_segment",
        suffixes=("_purch", f"_{event_name}"),
    )

    # Keep only valid pairs where precursor came before purchase
    merged = merged.loc[
        merged["event_ts_utc_purch"] > merged[f"event_ts_utc_{event_name}"]
    ].copy()

    # Enforce same-year rule
    same_year = (
        merged["event_ts_utc_purch"].dt.year
        == merged[f"event_ts_utc_{event_name}"].dt.year
    )
    merged = merged.loc[same_year]

    # Compute time differences in minutes
    merged["mins_between"] = (
        (merged["event_ts_utc_purch"] - merged[f"event_ts_utc_{event_name}"])
        .dt.total_seconds()
        / 60
    )

    # Compute average time gap per segment
    out = (
        merged.groupby("purchase_segment", as_index=False)
        .agg(
            avg_mins_between=("mins_between", "mean"),
            num_pairs=("mins_between", "size"),
        )
        .sort_values("avg_mins_between", ascending=False)
    )

    # Print summary
    mean_gap = out["avg_mins_between"].mean()
    print(f"{event_name}: {len(out)} segments analyzed, mean gap = {mean_gap:.2f} min")

    return out


def analyze_device(events, columnName, device_label):
    """Analyze recovery timing by device."""
    df = events[events[columnName] == device_label]

    # Group and summarize
    seg = (
        df.groupby(["customer_id", "purchase_segment"], as_index=False)
        .agg(
            start_ts=("event_ts_utc", "min"),
            end_ts=("event_ts_utc", "max"),
            status=("recovered", seg_status),
        )
    )
    seg = seg.sort_values(["customer_id", "start_ts"])
    seg = seg.groupby("customer_id", group_keys=False).apply(assign_groups)
    seg_kept = seg[seg["group_id"] > 0].copy()

    # Merge labels back
    df_labeled = df.merge(
        seg_kept[["customer_id", "purchase_segment", "group_id", "status"]],
        on=["customer_id", "purchase_segment"],
        how="inner",
    )

    # Build event sequences
    seq_df = (
        df_labeled.sort_values(["customer_id", "group_id", "event_ts_utc"])
        .groupby(["customer_id", "group_id"], as_index=False)
        .agg(
            segments=("purchase_segment", lambda s: list(pd.unique(s))),
            group_status=("status", "last"),
            start_ts=("event_ts_utc", "min"),
            end_ts=("event_ts_utc", "max"),
            num_events=("event_name", "size"),
            sequence=("event_name", list),
        )
    )
    seq_df["sequence_collapsed"] = seq_df["sequence"].apply(collapse_runs)

    # Compute durations
    seq_df["start_ts"] = pd.to_datetime(seq_df["start_ts"], utc=True, errors="coerce")
    seq_df["end_ts"] = pd.to_datetime(seq_df["end_ts"], utc=True, errors="coerce")
    seq_df["duration"] = seq_df["end_ts"] - seq_df["start_ts"]
    seq_df["duration_hours"] = seq_df["duration"].dt.total_seconds() / 3600
    dur = seq_df["duration_hours"].dropna().to_numpy()
    dur = dur[dur >= 0]

    # Print duration summary
    print(f"\nDuration summary (hours) for {device_label}")
    print(seq_df["duration_hours"].describe())

    # Plot ECDF
    p99 = np.percentile(dur, 99) if dur.size else 0.0
    xmax = p99 if np.isfinite(p99) and p99 > 0 else (dur.max() if dur.size else 1.0)
    x = np.sort(dur)
    y = np.arange(1, len(x) + 1) / len(x)
    plt.figure()
    plt.step(x, y, where="post")
    plt.xlim(0, 7000)
    plt.xlabel("Time to recovery (hours)")
    plt.ylabel("P(recovered by time ≤ t)")
    plt.title(f"Empirical chance of recovery over time ({device_label})")
    plt.grid(True)
    plt.show()

    # Compute landmark probabilities
    landmarks = [1, 6, 12, 24, 48, 72]
    landmark_probs = {f"{h}h": float((dur <= h).mean()) for h in landmarks}
    out = pd.Series(landmark_probs, name=device_label).to_frame().T
    return out


def seg_status(s: pd.Series) -> str:
    """Determine segment-level status."""
    sl = s.dropna().astype(str).str.lower()
    if (sl == "recovered").any():
        return "recovered"
    if (sl == "intermediate").any():
        return "intermediate"
    return "not applicable"


def assign_groups(g: pd.DataFrame) -> pd.DataFrame:
    """Assign group IDs for consecutive segments."""
    rec_flag = (g["status"] == "recovered").astype(int)
    grp = rec_flag.iloc[::-1].cumsum().iloc[::-1]
    g = g.copy()
    g["group_id"] = grp
    return g


def visualize_correlations(ratios, title):
    """Plot a bar chart showing category ratios."""
    ratios.plot(kind="bar", color="skyblue")
    plt.title(title)
    plt.xlabel("Category")
    plt.ylabel("Ratio")
    plt.ylim(0, 1)
    plt.show()


def plot_recovery_summary(recovery_summary, title_suffix):
    """Plot multi-line and grouped-bar recovery summaries."""
    cols = ["1h", "6h", "12h", "24h", "48h", "72h"]
    recovery_summary = recovery_summary[cols].astype(float)

    # Plot line chart
    x_hours = [1, 6, 12, 24, 48, 72]
    fig, ax = plt.subplots(figsize=(8, 5))
    for platform, row in recovery_summary.iterrows():
        ax.plot(x_hours, row.values, marker="o", linewidth=2, label=platform)
    ax.set_xticks(x_hours, cols)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time horizon")
    ax.set_ylabel("P(recovered by ≤ t)")
    ax.set_title(f"Recovery Probability {title_suffix}")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Platform", ncol=2, fontsize=9)
    plt.tight_layout()
    plt.show()

    # Plot grouped bar chart
    fig, ax = plt.subplots(figsize=(9, 5))
    bar_df = recovery_summary.T
    bar_df.plot(kind="bar", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time horizon")
    ax.set_ylabel("P(recovered by ≤ t)")
    ax.set_title(f"Recovery Probability {title_suffix} (Grouped Bars)")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def describe_events(out, summary_table, event_name):
    """Describe timing distribution for an event."""
    print(out["avg_mins_between"].describe())

    perc_75 = out["avg_mins_between"].describe()["75%"]
    print(f"75% of purchases occurred in < {math.ceil(perc_75)} minute(s).")

    perc_85 = out["avg_mins_between"].quantile(0.85)
    print(f"85% of purchases occurred in < {math.ceil(perc_85)} minute(s).")

    perc_90 = out["avg_mins_between"].quantile(0.9)
    print(f"90% of purchases occurred in < {math.ceil(perc_90)} minute(s).")

    summary_table.loc[summary_table["pattern"] == event_name, "75%"] = perc_75
    summary_table.loc[summary_table["pattern"] == event_name, "85%"] = perc_85

# Data Preparation

## Q1 & Q2 Preparation

preparation steps to analyze what leads to cart abandonment:

1.  Filter out irrelevant rows to cart abandonment. This is done by removing rows where abandonment is labeled false because no purchase was made and no items were added to the cart. 
2.  Add the event_page_name to the event_name column for the "page_view" and "button_click" events in order to further contexualize and differentiate between different types of page views and button clicks.
3.  flatten the table to create bins of event sequences in each segment. One row will correlate with one segment and contain all the sequence of events that lead to a successful purchase or an abandoned cart.

Notes: As we saw in the EDA, an overwhelming number of event_names were "page_view" and "button_click". I decided to add the event_page_name to the event_name column for these two actions in order to further contexualize and differentiate between different types of page views and button clicks.

In [0]:
# Change the datatypes
google_analytics['event_date'] = pd.to_datetime(google_analytics['event_date'])
google_analytics['event_ts_utc'] = pd.to_datetime(google_analytics['event_ts_utc'], utc=True)

# drop NAs in the dataset
google_analytics = google_analytics.dropna(subset=['abandoned'])

# Filter out inactive Google Analytics events
events_simplified = google_analytics[
    ~(
        (google_analytics['abandoned'] == False)
        & (google_analytics['false_by_purchase'] == 'no purchase')
    )
]

# Combine event name and page name for specific event types
mask = events_simplified['event_name'].isin(['page_view', 'button_click'])
events_simplified.loc[mask, 'event_name'] = (
    events_simplified.loc[mask, 'event_name'].fillna('')
    + ' - '
    + events_simplified.loc[mask, 'event_page_name'].fillna('')
)

# Sort events by purchase segment and timestamp for sequence mining
events_sequence_mining = events_simplified.sort_values(
    ['purchase_segment', 'event_ts_utc']
)

# Build event sequences for each purchase segment
seq_df_p1 = (
    events_sequence_mining.groupby('purchase_segment')
    .agg(
        customer_id=('customer_id', 'first'),
        abandoned=('abandoned', 'first'),
        sequence=('event_name', list),
        start_ts=('event_ts_utc', 'min'),
        end_ts=('event_ts_utc', 'max'),
        num_events=('event_name', 'size')
    )
    .reset_index()
)

# Apply run-collapsing transformation to sequences
seq_df_p1['sequence_collapsed'] = seq_df_p1['sequence'].apply(collapse_runs)

Steps to preparing for modeling what behaviors or conditions lead to cart recovery

1.  Figure out the current rate of cart recovery. 
2.  Filter only for rows that are labeled intermediate or recovery in the recovered column. 'recovery' indicates the segment where an abandoned cart was checked out, and 'intermediate' indicates the segments between an abandoned and recovered cart 
3.  flatten the table to create bins of event sequences in each segment. One row will correlate each recovery and it's preceding intermediate rows. 

In [0]:
# Compute recovery percentage among abandoned carts
events_flat = google_analytics.groupby(
    'purchase_segment', as_index=False
).last()

num_abandoned = events_flat['abandoned'].sum()
num_recovered = (events_flat['recovered'] == 'recovered').sum()

if num_abandoned > 0:
    pct = (num_recovered / num_abandoned) * 100
    print(f'{pct:.2f}% (recovered of abandoned carts)')
else:
    print('0.00% (recovered of abandoned carts; no abandoned carts found)')


# Select recovered or intermediate sessions and normalize event labels
events_recovered = google_analytics[
    google_analytics['recovered'].isin(['recovered', 'intermediate'])
].copy()

mask = events_recovered['event_name'].isin(['page_view', 'button_click'])
events_recovered.loc[mask, 'event_name'] = (
    events_recovered.loc[mask, 'event_name'].fillna('') + ' - ' +
    events_recovered.loc[mask, 'event_page_name'].fillna('')
)


# Aggregate segment-level timelines and statuses
ga = events_recovered.copy()

seg = (
    ga.groupby(['customer_id', 'purchase_segment'], as_index=False)
      .agg(
          start_ts=('event_ts_utc', 'min'),
          end_ts=('event_ts_utc', 'max'),
          status=('recovered', seg_status)
      )
)


# Order segments chronologically within each customer
seg = seg.sort_values(['customer_id', 'start_ts'])
seg = seg.groupby('customer_id', group_keys=False).apply(assign_groups)


# Keep only groups anchored by recovery (group_id > 0)
seg_kept = seg[seg['group_id'] > 0].copy()


# Attach group labels and statuses back to event-level data
ga_labeled = ga.merge(
    seg_kept[['customer_id', 'purchase_segment', 'group_id', 'status']],
    on=['customer_id', 'purchase_segment'],
    how='inner'
)


# Build per-group sequences and summary statistics
seq_df_p2 = (
    ga_labeled
      .sort_values(['customer_id', 'group_id', 'event_ts_utc'])
      .groupby(['customer_id', 'group_id'], as_index=False)
      .agg(
          segments=('purchase_segment', lambda s: list(pd.unique(s))),
          group_status=('status', 'last'),
          start_ts=('event_ts_utc', 'min'),
          end_ts=('event_ts_utc', 'max'),
          num_events=('event_name', 'size'),
          sequence=('event_name', list),
      )
)


# Collapse consecutive duplicate events in the sequence
seq_df_p2['sequence_collapsed'] = seq_df_p2['sequence'].apply(collapse_runs)

## Q3 & Q4 Preparation

This step extracts a clean mapping between each material ID and its associated trademark (brand) by removing duplicate pairs. Since the raw product catalog may contain repeated entries, this ensures a 1 to 1 lookup between material ids and their brand labels. This clean mapping will be used later during feature engineering to enrich transactional and abandoned cart records with brand level information, enabling downstream modeling to analyze abandonment patterns by product brand.

In [0]:
# Get unique material-to-trademark combinations
unique_combos = materials[["material_id", "trademark"]].drop_duplicates()

display(unique_combos.head())

This code above prepares raw Google Analytics event data for modeling by filtering out clearly inactive or irrelevant records and expanding the nested items field into a row per product structure. It goes through the serialized JSON lists into usable dictionaries, then extracts item_id and quantity attributes to enable product level analysis. Invalid entries are deleted, and item_id values are standardized to numeric formats, with non product identifiers discarded. We also inspect and quantify the imbalance in the abandoned target, which later informs us about the downsampling strategy to use. Finally, all remaining missing values are removed to ensure data integrity before training as Null data will prevent the model from running. This structured event level dataset allows downstream models to learn relationships between product behavior, device context, and abandonment outcomes.

In [0]:
events = google_analytics.copy()

# Remove inactive records based on specific conditions
inactive_mask = (
    (events['abandoned'] == False)
    & (events['false_by_purchase'] == 'no purchase')
    & (events['recovered'] == 'not applicable')
)

# Filter out inactive records
events = events[~inactive_mask]

# Normalize and parse the "items" column into lists of dictionaries
events['items'] = (
    events['items']
    .fillna('[]')
    .astype(str)
    .str.strip()
    .apply(
        lambda s: []
        if s in ('', '[]')
        else (json.loads(s) if '"' in s else ast.literal_eval(s))
    )
)

# Expand the "items" list so each element becomes its own row
events = events.explode('items', ignore_index=True)

# Extract "item_id" and "quantity" fields from each item dictionary
events['item_id'] = events['items'].map(
    lambda d: d.get('item_id') if isinstance(d, dict) else None
)
events['quantity'] = events['items'].map(
    lambda d: d.get('quantity') if isinstance(d, dict) else None
)

# Standardize data types for extracted columns
events['item_id'] = events['item_id'].astype('string')
events['quantity'] = pd.to_numeric(events['quantity'], errors='coerce').astype('Int64')

# Remove rows where "item_id" is missing
events = events[events['item_id'].notna()].reset_index(drop=True)

# Preview the transformed data
print(events.head())

# Drop the 'items' column now that its contents have been expanded
events = events.drop(columns=['items'])

# Inspect unique item IDs
events['item_id'].unique()

# Remove records with non-numeric item IDs starting with '01t'
events = events[~events['item_id'].str.startswith('01t')].reset_index(drop=True)

# Convert item_id column to integer type
events['item_id'] = events['item_id'].astype(int)

# Review class balance for the 'abandoned' target variable
events['abandoned'].value_counts(normalize=True)

# Note imbalance for downsampling consideration
# print(
#     'The majority class (abandoned = False) represents approximately 90%, '
#     'so we will downsample the True class to achieve a 2:1 ratio.'
# )

# Check for missing values before modeling
display(events.isnull().sum())

# Remove records with null values
post_eventns = events.dropna().reset_index(drop=True)

# # Display raw counts of the target variable
# display(post_eventns['abandoned'].value_counts())

# Display normalized class proportions
display(post_eventns['abandoned'].value_counts(normalize=True))

Here, we prepare the cleaned event level dataset for machine learning by engineering time based features, handling class imbalance, and constructing a preprocessing pipeline. First, timestamp columns are parsed into proper datetime formats and transformed into more model friendly attributes such as day of week and hour of activity, which can help capture behavioral patterns tied to cart abandonment timing. After removing null values, we define the target variable and select a set of device, event, and product related predictor features.

The data is then split into training and testing sets using stratification to preserve the original class proportions. Stratification is so essential for this as it causes the train and test sets to be simillar. Because abandonment is rare, we downsample the majority class to achieve a more balanced 2:1 ratio, improving the model's ability to learn minority class patterns. We also identify numerical and categorical columns, scaling numeric values and one hot encoding categories using ColumnTransformer. Finally, we create a cross validation subset from the downsampled training data to speed up hyperparameter tuning. Overall these preprocessing steps ensure that the training data is balanced, standardized, and ready for modeling.

In [0]:
# Copy and prepare dataframe
df_prep = post_eventns.copy()

# Parse timestamp columns
df_prep["event_date"] = pd.to_datetime(df_prep["event_date"], errors="coerce")
df_prep["event_ts_utc"] = pd.to_datetime(df_prep["event_ts_utc"], errors="coerce")

# Create time-based features
df_prep["dow"] = df_prep["event_date"].dt.dayofweek
df_prep["hour"] = df_prep["event_ts_utc"].dt.hour

# Drop nulls after feature creation
df_prep = df_prep.dropna().reset_index(drop=True)
# print("After dropna:", df_prep.shape)

# Define features and target
y = df_prep["abandoned"].astype(int)
feature_cols = [
    "event_name",
    "device_category",
    "device_mobile_brand_name",
    "device_operating_system",
    "event_page_name",
    "quantity",
    "dow",
    "hour",
]
X = df_prep[feature_cols]

# Split train and test data
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=seed
)

# Downsample majority class for balance
counts = y_train_full.value_counts()
maj_label = counts.idxmax()
min_label = counts.idxmin()
n_min = counts[min_label]
n_maj_target = 2 * n_min

maj_idx = y_train_full[y_train_full == maj_label].index
min_idx = y_train_full[y_train_full == min_label].index

rng = np.random.RandomState(seed)
maj_idx_down = rng.choice(maj_idx, size=min(n_maj_target, len(maj_idx)), replace=False)
down_idx = np.concatenate([min_idx, maj_idx_down])

X_train = X_train_full.loc[down_idx].reset_index(drop=True)
y_train = y_train_full.loc[down_idx].reset_index(drop=True)

print("Train class balance BEFORE:", counts.to_dict())
print("Train class balance AFTER: ", y_train.value_counts().to_dict())

# Define preprocessing pipeline
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

# Create CV subset for tuning
cv_frac = 0.6
X_train_cv, _, y_train_cv, _ = train_test_split(
    X_train, y_train, train_size=cv_frac, stratify=y_train, random_state=seed
)
print(f"Using {cv_frac*100:.0f}% of downsampled train for CV:", X_train_cv.shape)

## Q5 Preparation

This code prepares a comprehensive dataset that integrates customer interaction data, product attributes, and financial metrics to support modeling of the revenue effects of cart abandonment. It consolidates key behavioral signals from analytics events, such as cart additions and purchases, aligns them with standardized pricing and profitability information from sales data, and enriches them with product characteristics. Through this process, each item-level event is transformed into a structured, time-aware record that captures whether it was purchased or abandoned, along with its associated revenue and cost context. The resulting dataset provides a clean, analytically ready foundation for modeling the financial impact of cart abandonment on total revenue and product mix, enabling robust predictions and scenario analysis.

In [0]:
# Filter analytics events for cart and purchase logic
event_names_of_interest = {"add_to_cart", "purchase"}
events = google_analytics[
    google_analytics["event_name"].isin(event_names_of_interest)
]

events = events[
    (events["event_name"] != "add_to_cart")
    | (
        (events["event_name"] == "add_to_cart")
        & (events["abandoned"])
    )
].sort_values("event_ts_utc", ascending=True)

purchase_mask = events["event_name"] == "purchase"
events.loc[purchase_mask, "abandoned"] = False


# Build event dataset with sessions and key columns
events = assign_session_id(events).sort_values(
    ["customer_id", "event_ts_utc"], ascending=[False, False]
)
events = events[
    ["session_id", "customer_id", "items", "event_ts_utc", "abandoned"]
]

# Expand items and join material attributes
items = events.copy()
items["items"] = items["items"].map(parse_items)
items = items[items["items"].notna()]
items = items.explode("items", ignore_index=True)
items = items[items["items"].notna()]

items_norm = pd.json_normalize(items["items"])

base_cols = [
    c for c in items.drop(columns=["items"]).columns if c != "abandoned"
]
ordered_cols = base_cols + list(items_norm.columns) + ["abandoned"]

events_flat = pd.concat(
    [items.drop(columns=["items"]), items_norm],
    axis=1,
)[ordered_cols]

events_flat["item_id"] = events_flat["item_id"].astype(str)
materials["material_id"] = materials["material_id"].astype(str)

events_flat = (
    events_flat.merge(
        materials,
        how="left",
        left_on="item_id",
        right_on="material_id",
    )
    .drop(columns=["material_id"])
)

# Engineer reference prices at multiple temporal grains
sales["unit_profit"] = sales["profit"] / sales["physical_volume"]
sales["unit_cost"] = sales["unit_price"] - sales["unit_profit"]

sales["material_id"] = sales["material_id"].astype(str)
sales["posting_date"] = pd.to_datetime(sales["posting_date"])

sales["avg_unit_price"] = sales.groupby(
    ["material_id", "posting_date"]
)["unit_price"].transform("mean")
sales["avg_unit_profit"] = sales.groupby(
    ["material_id", "posting_date"]
)["unit_profit"].transform("mean")
sales["avg_unit_cost"] = sales.groupby(
    ["material_id", "posting_date"]
)["unit_cost"].transform("mean")

sales["year_week"] = sales["posting_date"].dt.strftime("%Y-%U")
sales["week_avg_unit_price"] = sales.groupby(
    ["material_id", "year_week"]
)["unit_price"].transform("mean")
sales["week_avg_unit_profit"] = sales.groupby(
    ["material_id", "year_week"]
)["unit_profit"].transform("mean")
sales["week_avg_unit_cost"] = sales.groupby(
    ["material_id", "year_week"]
)["unit_cost"].transform("mean")

sales["year_month"] = sales["posting_date"].dt.strftime("%Y-%m")
sales["month_avg_unit_price"] = sales.groupby(
    ["material_id", "year_month"]
)["unit_price"].transform("mean")
sales["month_avg_unit_profit"] = sales.groupby(
    ["material_id", "year_month"]
)["unit_profit"].transform("mean")
sales["month_avg_unit_cost"] = sales.groupby(
    ["material_id", "year_month"]
)["unit_cost"].transform("mean")

sales["year"] = sales["posting_date"].dt.strftime("%Y")
sales["year_avg_unit_price"] = sales.groupby(
    ["material_id", "year"]
)["unit_price"].transform("mean")
sales["year_avg_unit_profit"] = sales.groupby(
    ["material_id", "year"]
)["unit_profit"].transform("mean")
sales["year_avg_unit_cost"] = sales.groupby(
    ["material_id", "year"]
)["unit_cost"].transform("mean")

# Create a de-duplicated price view with effective price backfill, including posting_date
price_view = sales[
    [
        "material_id",
        "posting_date",
        "unit_price",
        "avg_unit_price",
        "week_avg_unit_price",
        "month_avg_unit_price",
        "year_avg_unit_price",
        "unit_profit",
        "avg_unit_profit",
        "week_avg_unit_profit",
        "month_avg_unit_profit",
        "year_avg_unit_profit",
        "unit_cost",
        "avg_unit_cost",
        "week_avg_unit_cost",
        "month_avg_unit_cost",
        "year_avg_unit_cost",
    ]
].drop_duplicates()

price_view["effective_unit_price"] = (
    price_view[
        [
            "avg_unit_price",
            "week_avg_unit_price",
            "month_avg_unit_price",
            "year_avg_unit_price",
        ]
    ]
    .bfill(axis=1)
    .iloc[:, 0]
    .round(2)
)

price_view["effective_unit_profit"] = (
    price_view[
        [
            "avg_unit_profit",
            "week_avg_unit_profit",
            "month_avg_unit_profit",
            "year_avg_unit_profit",
        ]
    ]
    .bfill(axis=1)
    .iloc[:, 0]
    .round(2)
)

price_view["effective_unit_cost"] = (
    price_view[
        [
            "avg_unit_cost",
            "week_avg_unit_cost",
            "month_avg_unit_cost",
            "year_avg_unit_cost",
        ]
    ]
    .bfill(axis=1)
    .iloc[:, 0]
)

# Drop engineered avg columns for price, profit, and cost
cols_to_drop = [
    "avg_unit_price", "week_avg_unit_price", "month_avg_unit_price", "year_avg_unit_price",
    "avg_unit_profit", "week_avg_unit_profit", "month_avg_unit_profit", "year_avg_unit_profit",
    "avg_unit_cost", "week_avg_unit_cost", "month_avg_unit_cost", "year_avg_unit_cost"
]
price_view = price_view.drop(columns=cols_to_drop)

# Group by material_id and posting_date, set effective columns to mean of each
price_view = (
    price_view
    .groupby(["material_id", "posting_date"], as_index=False)[
        ["effective_unit_price", "effective_unit_profit", "effective_unit_cost"]
    ]
    .mean()
    .round(2)
)

# Join effective_unit_price to events_flat by item_id=material_id and event_ts_utc (date)=posting_date
events_flat["event_date"] = pd.to_datetime(events_flat["event_ts_utc"]).dt.date
price_view["posting_date"] = pd.to_datetime(price_view["posting_date"]).dt.date

full_events = events_flat.merge(
    price_view[[
        "material_id", "posting_date", "effective_unit_price",
        "effective_unit_profit", "effective_unit_cost"
    ]],
    left_on=["item_id", "event_date"],
    right_on=["material_id", "posting_date"],
    how="inner"
).drop(columns=["material_id", "posting_date"])

# Clean quantity to an integer vector
qty = (
    pd.to_numeric(full_events["quantity"], errors="coerce")
    .fillna(1)
    .astype("int64")
)

# Build repeated row positions with NumPy (compact and fast)
pos = np.repeat(np.arange(len(full_events), dtype=np.int64), qty.to_numpy())

# Take rows by position to avoid index alignment overhead
item_activity_data = (
    full_events
    .take(pos)
    .reset_index(drop=True)
    .drop(columns=["quantity"])
)
item_activity_data["item_activity_id"] = item_activity_data.index.astype("int64")

item_activity_data = item_activity_data[
    [
        "item_activity_id",
        "session_id",
        "customer_id",
        "event_ts_utc",
        "item_id",
        "trademark",
        "flavor",
        "beverage_category",
        "packaging_type",
        "packaging_size",
        "effective_unit_price",
        "effective_unit_profit",
        "effective_unit_cost",
        "abandoned"
    ]
].sort_values(["customer_id", "event_ts_utc", "item_id"])

display(item_activity_data.head())
# item_activity_data.to_csv("/Volumes/workspace/default/capstone_data/item_activity.csv", index=False)

Next we transform the detailed item-level activity data into a structured, time-series summary that quantifies both realized and potential financial outcomes. It standardizes timestamps, converts financial values to numeric form, and calculates realized versus potential revenue and profit to isolate the monetary value of abandoned carts. The data is then aggregated by day to compute total revenue, profit, and abandonment-related losses, along with daily abandonment rates and transaction volumes. The resulting dataset provides a clean temporal view of financial performance and customer behavior, enabling downstream modeling of revenue impact trends, profitability forecasting, and the drivers of cart abandonment over time.

In [0]:
item_activity_copy = item_activity_data.copy()

# Convert timestamp and boolean columns
item_activity_copy["event_ts_utc"] = pd.to_datetime(item_activity_copy["event_ts_utc"], utc=True, errors="coerce")
item_activity_copy["abandoned"] = item_activity_copy["abandoned"].astype(bool)

# Convert monetary columns to numeric
for col in ["effective_unit_price", "effective_unit_profit", "effective_unit_cost"]:
    item_activity_copy[col] = pd.to_numeric(item_activity_copy[col], errors="coerce")

# Compute realized and potential revenue
item_activity_copy["realized_revenue"] = np.where(item_activity_copy["abandoned"], 0.0, item_activity_copy["effective_unit_price"])
item_activity_copy["potential_revenue"] = item_activity_copy["effective_unit_price"]
item_activity_copy["abandoned_revenue"] = item_activity_copy["potential_revenue"] - item_activity_copy["realized_revenue"]

# Compute realized and potential profit
item_activity_copy["realized_profit"] = np.where(item_activity_copy["abandoned"], 0.0, item_activity_copy["effective_unit_profit"])
item_activity_copy["potential_profit"] = item_activity_copy["effective_unit_profit"]
item_activity_copy["abandoned_profit"] = item_activity_copy["potential_profit"] - item_activity_copy["realized_profit"]

# Create calendar-based daily aggregation
item_activity_copy["event_date"] = item_activity_copy["event_ts_utc"].dt.tz_convert("UTC").dt.floor("D")

# Aggregate daily revenue and profit metrics
daily = (
    item_activity_copy.groupby("event_date", as_index=False)[
        [
            "realized_revenue",
            "potential_revenue",
            "abandoned_revenue",
            "realized_profit",
            "potential_profit",
            "abandoned_profit",
        ]
    ]
    .sum()
    .sort_values("event_date")
)

# Compute daily abandonment rate and total item counts
daily_counts = item_activity_copy.groupby("event_date")["abandoned"].agg(["mean", "size"]).reset_index()
daily = daily.merge(daily_counts, on="event_date", how="left")
daily = daily.rename(columns={"mean": "abandon_rate", "size": "items_cnt"})

# Display preview of the daily summary
display(daily.head())

## Q6 Preparation

The following code will examine post-abandonment patterns, so the data will be filtered to include only customers who abandoned a cart, and only their interactions/orders after their first abandonment. We will merge google analytics data with orders data and format each row to represent a single date. Each date will contain a binary indicator for either a purchase, abandonment, or order.

In [0]:
ga_with_abandoned = google_analytics.copy()

# Fill missing abandoned flags with False
ga_with_abandoned["abandoned"] = (
    ga_with_abandoned["abandoned"]
    .astype(bool)
    .fillna(False)
)

# Sort by customer and event timestamp
ga_with_abandoned = ga_with_abandoned.sort_values(
    by=["customer_id", "event_ts_utc"], ascending=[True, True]
)

# Identify customers who abandoned at least once
abandoned_customers = ga_with_abandoned.loc[
    ga_with_abandoned["abandoned"] == True, "customer_id"
].unique()

# Count unique abandoned customers
num_abandoned_customers = len(abandoned_customers)
print(f"Number of unique customers who abandoned: {num_abandoned_customers}")

# Filter to include only customers who have abandoned
ga_abandoned_customers = ga_with_abandoned[
    ga_with_abandoned["customer_id"].isin(abandoned_customers)
]

# Sort filtered data
ga_abandoned_customers = ga_abandoned_customers.sort_values(
    ["customer_id", "event_ts_utc"], ascending=[True, True]
)

# Find the first abandonment timestamp per customer
first_abandoned_ts = (
    ga_abandoned_customers[ga_abandoned_customers["abandoned"] == True]
    .groupby("customer_id")["event_ts_utc"]
    .min()
    .reset_index()
    .rename(columns={"event_ts_utc": "first_abandoned_ts"})
)

# Merge first abandonment timestamp back to full table
ga_after_first_abandoned = ga_abandoned_customers.merge(
    first_abandoned_ts, on="customer_id", how="left"
)

# Keep only events after first abandonment
ga_after_first_abandoned = ga_after_first_abandoned[
    ga_after_first_abandoned["event_ts_utc"]
    >= ga_after_first_abandoned["first_abandoned_ts"]
].drop(columns=["first_abandoned_ts"])

# Sort by customer and timestamp
ga_after_first_abandoned = ga_after_first_abandoned.sort_values(
    ["customer_id", "event_ts_utc"], ascending=[True, True]
)

# Convert date columns to datetime
ga_after_first_abandoned["event_date"] = pd.to_datetime(
    ga_after_first_abandoned["event_date"]
)
orders["created_ts_utc"] = pd.to_datetime(orders["created_ts_utc"])
orders["created_date"] = orders["created_ts_utc"].dt.date
orders["created_date"] = pd.to_datetime(orders["created_date"])

# Build GA events table
ga_events = (
    ga_after_first_abandoned.loc[
        ga_after_first_abandoned["event_date"]
        >= ga_after_first_abandoned.groupby("customer_id")["event_date"].transform("min"),
        [
            "customer_id",
            "device_category",
            "device_mobile_brand_name",
            "device_operating_system",
            "event_date",
            "abandoned",
            "event_name",
        ],
    ]
    .assign(
        is_abandon=lambda df: df["abandoned"] == True,
        is_purchase=lambda df: df["event_name"].str.lower().str.contains("purchase"),
        is_order=False,
    )
)

# Build order events table
order_events = (
    orders[
        [
            "customer_id",
            "material_id",
            "sales_office_id",
            "order_quantity",
            "order_type",
            "created_date",
        ]
    ]
    .rename(columns={"created_date": "event_date"})
    .assign(
        device_category=None,
        device_mobile_brand_name=None,
        device_operating_system=None,
        abandoned=False,
        event_name="order_created",
        is_abandon=False,
        is_purchase=False,
        is_order=True,
    )
)

# Combine GA and order events
combined_events = pd.concat([ga_events, order_events], ignore_index=True)

# Sort combined events
combined_events = combined_events.sort_values(["customer_id", "event_date"]).reset_index(
    drop=True
)

# Assign order_type for purchases
combined_events.loc[combined_events["is_purchase"], "order_type"] = "mycoke360"

# Assign order_type for abandons
combined_events.loc[combined_events["is_abandon"], "order_type"] = "none"

# Merge combined events with customer table
combined_with_customer = combined_events.merge(
    customer,
    on="customer_id",
    how="left",
)

# Create dataframe of abandoned customers
abandoned_df = pd.DataFrame({"customer_id": abandoned_customers})

# Merge combined events with customer table again
combined_with_customer = combined_events.merge(
    customer,
    on="customer_id",
    how="left",
)

# Ensure all abandoned customers are retained
combined_with_customer = abandoned_df.merge(
    combined_with_customer,
    on="customer_id",
    how="left",
)

After confirming the first abandonment date per customer, we will see if they had any orders. If not, we will conclude that they have churned. This will be flagged and added back to the original dataframe. 

In [0]:
# Convert event_date to datetime
combined_with_customer["event_date"] = pd.to_datetime(combined_with_customer["event_date"])

# Compute first abandon date per customer
first_abandon_date = (
    combined_with_customer[combined_with_customer["is_abandon"] == True]
    .groupby("customer_id")["event_date"]
    .min()
    .reset_index()
    .rename(columns={"event_date": "first_abandon_date"})
)

# Merge first abandon date into main dataframe
combined_with_customer = combined_with_customer.merge(first_abandon_date, on="customer_id", how="left")

# Identify post-abandon purchases or orders
post_abandon = combined_with_customer[
    ((combined_with_customer["is_purchase"] == True) |
     (combined_with_customer["is_order"] == True)) &
    (combined_with_customer["event_date"] >= combined_with_customer["first_abandon_date"])
]

# Create churn flag based on absence of post-abandon activity
churn_flag = combined_with_customer[["customer_id"]].drop_duplicates().copy()
churn_flag["churned"] = ~churn_flag["customer_id"].isin(post_abandon["customer_id"])

# Calculate churn counts
churn_counts = churn_flag["churned"].value_counts()

# Calculate churn proportions
churn_proportions = churn_flag["churned"].value_counts(normalize=True)

# print("Counts:")
# print(churn_counts)
# print("\nProportions:")
# print(churn_proportions)

# Merge churn flag data into main dataframe
combined_with_customer = combined_with_customer.merge(
    churn_flag,
    on="customer_id",
    how="left"
)

Next we will filter the data to find customers who didn't necessarily churn, but didnt order through mycoke360 after their initial instance of abandonment.

In [0]:
# Keep events after first abandon
after_abandon = combined_with_customer[
    combined_with_customer['event_date'] >= combined_with_customer['first_abandon_date']
]

# Aggregate per customer
customer_flags = after_abandon.groupby('customer_id').agg(
    ordered_any=('is_order', 'any'),
    purchased_any=('is_purchase', 'any'),
    abandoned_any=('is_abandon', 'any')
).reset_index()

# Filter: ordered but never purchased or abandoned again
ordered_only_customers = customer_flags[
    (customer_flags['ordered_any'] == True) &
    (customer_flags['purchased_any'] == False)
]

print(f"Customers who ordered but never purchased or abandoned again: {len(ordered_only_customers)}")

Then we will track a customer's order methods to see if their first instance of abandonment indicated a switch in preference. 

In [0]:
# Split events into before and after first abandonment
before_abandon = combined_with_customer[
    combined_with_customer["event_date"] < combined_with_customer["first_abandon_date"]
]

after_abandon = combined_with_customer[
    combined_with_customer["event_date"] >= combined_with_customer["first_abandon_date"]
]

# Identify unique order types before abandonment
before_types = (
    before_abandon[before_abandon["is_order"]]
    .groupby("customer_id")["order_type"]
    .unique()
    .reset_index()
    .rename(columns={"order_type": "before_order_types"})
)

# Identify unique order types after abandonment
after_types = (
    after_abandon[after_abandon["is_order"]]
    .groupby("customer_id")["order_type"]
    .unique()
    .reset_index()
    .rename(columns={"order_type": "after_order_types"})
)

# Merge before and after order types
channel_switch = before_types.merge(after_types, on="customer_id", how="inner")

# Flag customers who switched order channels
channel_switch["switched_channel"] = channel_switch.apply(switched, axis=1)

# Filter switched customers
switched_customers = channel_switch[channel_switch["switched_channel"]]
print(f"Customers who switched order channel after first abandonment: {len(switched_customers)}")

Finally, we will indentify each unique customer that switched order methods and capture their preferences, pre and post abandonment.

In [0]:
# Build switch pairs for customers who changed order types
switch_pairs = []

for _, row in switched_customers.iterrows():
    before = set(row["before_order_types"])
    after = set(row["after_order_types"])
    new_channels = after - before

    # Record transitions from previous to new order types
    for b, a in product(before, new_channels):
        switch_pairs.append((b, a))

# Create dataframe of channel switches
switch_df = pd.DataFrame(switch_pairs, columns=["from_channel", "to_channel"])

# Summarize most frequent switch patterns
switch_summary = (
    switch_df.value_counts()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)

# Modeling & Performance

## Q1 & Q2 Modeling

Steps to assess what leads to cart abandonment:

1.  Perform sequence mining on to discover the top first events, last events, bigrams, trigrams, and full sequences that lead to cart abandonment. 
2.  Extract and Analyze the top 10 events or sequence of events that lead to the highest percentage of cart abandonment and append them to a single table for analysis.
3.  Use the results of the time-distance-from-purchase analysis to inform when Swire should reach out to customers after each event is clicked to remind them they have items in their cart. 

In [0]:
# Split the dataframe into abandoned and completed sessions
mask_abd = seq_df_p1['abandoned'].astype(bool)
seqs_abd = seq_df_p1.loc[mask_abd, 'sequence']
seqs_cmp = seq_df_p1.loc[~mask_abd, 'sequence']

# Compute first and last events for each session type
first_abd = top_counts(seqs_abd.str[0], top_n=10)
first_cmp = top_counts(seqs_cmp.str[0], top_n=10)
last_abd = top_counts(seqs_abd.str[-1], top_n=10)
last_cmp = top_counts(seqs_cmp.str[-1], top_n=10)

first_events = compare_tables(first_abd, first_cmp)
last_events = compare_tables(last_abd, last_cmp)

# Compute top full sequences (lists converted to tuples)
full_abd = top_counts(
    seqs_abd.map(lambda s: tuple(s) if isinstance(s, list) else ()),
    top_n=10
)
full_cmp = top_counts(
    seqs_cmp.map(lambda s: tuple(s) if isinstance(s, list) else ()),
    top_n=10
)
full_sequences = compare_tables(full_abd, full_cmp)

# Compute bigrams and trigrams across sessions
bigrams_abd = top_ngrams_df(seqs_abd, n=2, top_n=10)
bigrams_cmp = top_ngrams_df(seqs_cmp, n=2, top_n=10)
trigrams_abd = top_ngrams_df(seqs_abd, n=3, top_n=10)
trigrams_cmp = top_ngrams_df(seqs_cmp, n=3, top_n=10)

bigrams = compare_tables(bigrams_abd, bigrams_cmp)
trigrams = compare_tables(trigrams_abd, trigrams_cmp)

# Display summaries for inspection
print('\nTop FIRST events (abandoned vs completed):')
display(first_events.head(15))

print('\nTop LAST events (abandoned vs completed):')
display(last_events.head(15))

print('\nTop FULL sequences (abandoned vs completed):')
display(full_sequences.head(10))

print('\nTop BIGRAMS (abandoned vs completed):')
display(bigrams.head(15))

print('\nTop TRIGRAMS (abandoned vs completed):')
display(trigrams.head(15))

The top first events aren't very helpful as it indicates session_starts, dahsboards visits, and purchase success pages from previous purchases. Not much insight can be gleaned from this

The top last events are where the most information can be collected. As we can see above, 70% of the instances of cart abandonment we identified above can be explained by these features. Most notably, update_cart. 

The full sequences, bigrams, and trigams are not very helpful in predicting cart abandonment. The full sequences are too unique as there isn't a single duplicate pattern identified. The bigrams and trigrams are mostly standard website clicks and don't represent enough of the abandoned dataset.

In [0]:
# Create a summary table from the last events containing abandonment percentages
summary_table = last_events.loc[
    last_events['percent_abandoned'].notna(),
    ['pattern', 'percent_abandoned']
].copy()

# Add empty columns for the 75% and 85% thresholds
summary_table['75%'] = ''
summary_table['85%'] = ''

# Display the first 10 rows of the summary table
summary_table.head(n=10)

This summary will be used later to append the 75% and 85% threshold for hours to purchase. As such, it requires no interpretation now.

In [0]:
# Create the purchases table
purchases = events_simplified.loc[
    (events_simplified['event_name'] == 'purchase')
    & (events_simplified['abandoned'] != True)
    & (events_simplified['recovered'] != 'recovered'),
    ['event_name', 'event_ts_utc', 'purchase_segment']
]

# Analyze update_cart events
update_carts = events_simplified.loc[
    events_simplified['event_name'] == 'update_cart',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, update_carts, 'update_cart')
describe_events(out, summary_table, 'update_cart')

# Analyze remove_from_cart events
remove_from_cart = events_simplified.loc[
    events_simplified['event_name'] == 'remove_from_cart',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, remove_from_cart, 'remove_from_cart')
describe_events(out, summary_table, 'remove_from_cart')

# Analyze view_item_list events
view_item_list = events_simplified.loc[
    events_simplified['event_name'] == 'view_item_list',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, view_item_list, 'view_item_list')
describe_events(out, summary_table, 'view_item_list')

# Analyze page_view - Unknown Page events
unknown = events_simplified.loc[
    events_simplified['event_name'] == 'page_view - Unknown Page',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, unknown, 'page_view - Unknown Page')
describe_events(out, summary_table, 'page_view - Unknown Page')

# Analyze page_view - Mycoke Dashboard events
mycoke_dashboard = events_simplified.loc[
    events_simplified['event_name'] == 'page_view - Mycoke Dashboard',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, mycoke_dashboard,
                               'page_view - Mycoke Dashboard')
describe_events(out, summary_table, 'page_view - Mycoke Dashboard')

# Analyze user_engagement events
user_engagement = events_simplified.loc[
    events_simplified['event_name'] == 'user_engagement',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, user_engagement, 'user_engagement')
describe_events(out, summary_table, 'user_engagement')

# Analyze button_click - Mycoke Orders - Cart events
orders_cart = events_simplified.loc[
    events_simplified['event_name'] == 'button_click - Mycoke Orders - Cart',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, orders_cart,
                               'button_click - Mycoke Orders - Cart')
describe_events(out, summary_table, 'button_click - Mycoke Orders - Cart')

# Analyze page_view - Mycoke Orders events
orders = events_simplified.loc[
    events_simplified['event_name'] == 'page_view - Mycoke Orders',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, orders, 'page_view - Mycoke Orders')
describe_events(out, summary_table, 'page_view - Mycoke Orders')

# Analyze button_click - Mycoke Orders - Checkout: Review Order events
review_order = events_simplified.loc[
    events_simplified['event_name']
    == 'button_click - Mycoke Orders - Checkout: Review Order',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(
    purchases, review_order, 'button_click - Mycoke Orders - Checkout: Review Order'
)
describe_events(out, summary_table,
                'button_click - Mycoke Orders - Checkout: Review Order')

# Analyze proceed_to_checkout events
proceed_to_checkout = events_simplified.loc[
    events_simplified['event_name'] == 'proceed_to_checkout',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, proceed_to_checkout,
                               'proceed_to_checkout')
describe_events(out, summary_table, 'proceed_to_checkout')

# Summarize event metrics and display results
summary_table[['75%', '85%']] = (
    summary_table[['75%', '85%']].astype(float) / 60
).round(0)
display(summary_table.head(n=10))

# Calculate 75% threshold reduction potential
total_75 = summary_table.loc[
    summary_table['75%'] < 24, 'percent_abandoned'
].sum()
print(
    "Please refer to the chart above to see which event_names correlate "
    "with a 75% chance of purchase within one day of happening. "
    "We can potentially reduce cart abandonment by",
    round(total_75, 0),
    "percent if we reach out to customers who enter any of these events "
    "and don't purchase within a day.",
)

# Calculate 85% threshold reduction potential
total_85 = summary_table.loc[
    summary_table['85%'] < 24, 'percent_abandoned'
].sum()
print(
    "Please refer to the chart above to see which event_names correlate "
    "with an 85% chance of purchase within one day of happening. "
    "We can potentially reduce cart abandonment by",
    round(total_85, 0),
    "percent if we reach out to customers who enter any of these events "
    "and don't purchase within a day.",
)

Events that on average have a 75% of a purchase taking place within one day of being executed. These are the last events in 70% of abandoned cart segments:
1. update_cart
2. remove_from_cart
3. view_item_list
4. page_view - Unknown Page
5. page_view - Mycoke Dashboard	
6. button_click - Mycoke Orders - Cart
7. page_view - Mycoke Orders
8. button_click - Mycoke Orders - Checkout: Review Order
9. proceed_to_checkout

Events that on average have a 85% of a purchase taking place within one day of being executed. These are the last events in 46% of abandoned cart segments:
1. update_cart
2. remove_from_cart
3. button_click - Mycoke Orders - Cart
4. button_click - Mycoke Orders - Checkout: Review Order
5. proceed_to_checkout

In [0]:
# Select only recovered segments
if 'group_status' in seq_df_p2.columns:
    mask_rec = (
        seq_df_p2['group_status']
        .astype(str)
        .str.lower()
        .eq('recovered')
    )
elif 'recovered' in seq_df_p2.columns:
    mask_rec = (
        seq_df_p2['recovered']
        .astype(str)
        .str.lower()
        .eq('recovered')
    )
else:
    raise KeyError(
        "seq_df_p2 requires either 'group_status' or 'recovered' column "
        "to identify recovered segments."
    )

seqs_rec = seq_df_p2.loc[mask_rec, 'sequence']

# Compute top first and last events
first_rec = top_counts(seqs_rec.str[0], top_n=15)
last_rec = top_counts(seqs_rec.str[-1], top_n=15)

# Compute top full sequences
full_rec = top_counts(
    seqs_rec.map(lambda s: tuple(s) if isinstance(s, list) else ()),
    top_n=10,
)

# Compute top bigrams and trigrams
bigrams_rec = top_ngrams_df(seqs_rec, n=2, top_n=15)
trigrams_rec = top_ngrams_df(seqs_rec, n=3, top_n=15)

# Display analysis results
print("\nTop FIRST events (recovered only):")
display(first_rec.head(15))

print("\nTop LAST events (recovered only):")
display(last_rec.head(15))

print("\nTop FULL sequences (recovered only):")
display(full_rec.head(10))

print("\nTop BIGRAMS (recovered only):")
display(bigrams_rec.head(15))

print("\nTop TRIGRAMS (recovered only):")
display(trigrams_rec.head(15))

As we can see from the sequence mining results above, there isn't much insight we can gather. the results are similar to the abandonment results, except the last events are less insightful since 92% are purchase (which makes sense). We should focus on understanding time as a factor for cart recovery

In [0]:
# Convert timestamp columns to datetime with UTC timezone
seq_df_p2['start_ts'] = pd.to_datetime(
    seq_df_p2['start_ts'],
    utc=True,
    errors='coerce'
)
seq_df_p2['end_ts'] = pd.to_datetime(
    seq_df_p2['end_ts'],
    utc=True,
    errors='coerce'
)

# Compute duration as a timedelta
seq_df_p2['duration'] = seq_df_p2['end_ts'] - seq_df_p2['start_ts']

# Derive duration in various units
seq_df_p2['duration_hours'] = seq_df_p2['duration'].dt.total_seconds() / 3600
seq_df_p2['duration_days'] = seq_df_p2['duration'].dt.total_seconds() / 86400
seq_df_p2['duration_minutes'] = seq_df_p2['duration'].dt.total_seconds() / 60

# Display descriptive statistics for duration (in hours)
print("\nDuration summary (hours)")
print(seq_df_p2['duration_hours'].describe())

# Summarize duration by group_status
summary = (
    seq_df_p2.groupby('group_status')['duration_hours']
    .describe(percentiles=[0.25, 0.5, 0.75])
    .round(2)
)
print("\nDuration by group_status")
print(summary)

# Identify the five longest recovered sequences
longest = seq_df_p2.nlargest(
    5,
    'duration_hours'
)[['customer_id', 'group_id', 'duration_hours', 'num_events']]

# Identify the five shortest recovered sequences
shortest = seq_df_p2.nsmallest(
    5,
    'duration_hours'
)[['customer_id', 'group_id', 'duration_hours', 'num_events']]

print("\nLongest recovered sequences:")
display(longest)

print("\nShortest recovered sequences:")
display(shortest)

As we can see above, 50% of cart recovery occurs in the first few hours after abandonment. After the first few hours, recovery time becomes on an order of days/weeks not hours.

In [0]:
# Compute recovery duration in hours
seq_df_p2['start_ts'] = pd.to_datetime(seq_df_p2['start_ts'], utc=True, errors='coerce')
seq_df_p2['end_ts'] = pd.to_datetime(seq_df_p2['end_ts'], utc=True, errors='coerce')
seq_df_p2['duration_hours'] = (
    (seq_df_p2['end_ts'] - seq_df_p2['start_ts']).dt.total_seconds() / 3600
)

# Filter valid durations
durations = seq_df_p2.loc[seq_df_p2['duration_hours'] > 0, 'duration_hours']

# Calculate the 90th percentile of duration
p90 = durations.quantile(0.90)

# Plot histogram with 90th percentile line
plt.figure(figsize=(8, 4))
sns.histplot(durations, bins=30, kde=True, color='skyblue')
plt.axvline(
    p90,
    color='red',
    linestyle='--',
    linewidth=2,
    label=f"90th percentile = {p90:.1f} hrs",
)
plt.title("Distribution of Recovery Durations (Hours)", fontsize=13, weight='bold')
plt.xlabel("Duration (hours)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

# Recompute histogram statistics
durations = seq_df_p2.loc[seq_df_p2['duration_hours'] > 0, 'duration_hours']
counts, bin_edges = np.histogram(durations, bins=30)

# Extract first bin statistics
first_bin_count = int(counts[0])
first_bin_start = bin_edges[0]
first_bin_end = bin_edges[1]
first_bin_range = (first_bin_start, first_bin_end)

# Compute percentage of observations in the first bin
total_obs = int(counts.sum())
first_bin_pct = (first_bin_count / total_obs * 100) if total_obs else 0.0

# Print summary statistics
print(f"Total observations: {total_obs:,}")
print(
    f"Number of observations in first bin: {first_bin_count:,} "
    f"({first_bin_pct:.2f}% of total)"
)
print(
    f"Duration range of first bin: "
    f"{first_bin_range[0]:.2f} to {first_bin_range[1]:.2f} hours"
)

# Visualize histogram with first bin boundary
plt.figure(figsize=(8, 4))
sns.histplot(durations, bins=30, kde=True, color='skyblue')
plt.axvline(
    first_bin_end,
    color='green',
    linestyle='--',
    linewidth=2,
    label=(
        f"End of 1st bin = {first_bin_end:.1f} hrs\n"
        f"{first_bin_pct:.1f}% of data"
    ),
)
plt.title(
    "Histogram of Recovery Durations with First Bin Boundary",
    fontsize=13,
    weight='bold',
)
plt.xlabel("Duration (hours)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
# Extract valid duration values
dur = seq_df_p2['duration_hours'].dropna().to_numpy()
dur = dur[dur >= 0]

# Compute percentile cap for plotting range
p99 = np.percentile(dur, 99) if dur.size else 0.0
xmax = p99 if np.isfinite(p99) and p99 > 0 else (dur.max() if dur.size else 1.0)

# Compute empirical cumulative distribution function
x = np.sort(dur)
y = np.arange(1, len(x) + 1) / len(x)

# Plot ECDF of recovery duration
plt.figure()
plt.step(x, y, where="post")
plt.xlim(0, xmax)
plt.xlabel("Time to recovery (hours)")
plt.ylabel("P(recovered by time ≤ t)")
plt.title("Empirical chance of recovery over time")
plt.grid(True)
plt.show()

# Compute quick probability landmarks
landmarks = [1, 6, 12, 24, 48, 72]
landmark_probs = {f"{h}h": float((dur <= h).mean()) for h in landmarks}
pd.Series(landmark_probs, name="P(recovered by ≤ t)").to_frame()

We can see very clearly that there is a steep chance of prolongued abandonment over time. Summarized results below:
    50% of carts are recovered in 2 hours.
    65% of carts are recovered in 3 days.
    75% of carts are recovered in 168 hours, or 7 days.
    80% of carts are recovered in 235 hours, or 10 days.
    90% of carts are recovered in 674 hours, or 28 days. 
    
It appears that the steep drop off in recovery over time shows that abandonment is intentional. Customers either return to their carts, or they move on and wait until they need inventory. 

In [0]:
# Analyze recovery probability by device category
desktop_tbl = analyze_device(events_recovered, 'device_category', 'Desktop')
mobile_tbl = analyze_device(events_recovered, 'device_category', 'Mobile')
tablet_tbl = analyze_device(events_recovered, 'device_category', 'Tablet')

# Combine results from all device categories into one summary table
recovery_summary = pd.concat([desktop_tbl, mobile_tbl, tablet_tbl])

# Display the combined recovery probability summary
print("\nCombined Recovery Probability Table")
display(recovery_summary)

As we can see from the summarized table above, desktop recovers the slowest and tablet users recover the fastest. However, after the 3 day mark, we don't see much of a difference between the 3 devices. 

One thing to note, is that if a tablet user has yet to recover by 6 hours, they won't for 3 days. Whereas desktop and mobile users increase recovery slowly over time.

In [0]:
# Plot the recovery summary by device category
plot_recovery_summary(recovery_summary, 'device_category')

As we can see above, tablet users have the highest chances of recovery, with mobile users in the middle, and desktop having the lowest chance. The chances of recovery diminish very quickly across all devices as the rate barely increases from 12h to 72h.

In [0]:
# Analyze recovery probability by device brand
google = analyze_device(events_recovered, 'device_mobile_brand_name', 'Google')
apple = analyze_device(events_recovered, 'device_mobile_brand_name', 'Apple')
microsoft = analyze_device(events_recovered, 'device_mobile_brand_name', 'Microsoft')
samsung = analyze_device(events_recovered, 'device_mobile_brand_name', 'Samsung')
mozilla = analyze_device(events_recovered, 'device_mobile_brand_name', 'Mozilla')
other = analyze_device(events_recovered, 'device_mobile_brand_name', 'Other')

# Combine all device results into a single summary table
recovery_summary = pd.concat(
    [google, apple, microsoft, samsung, mozilla, other]
).reset_index(drop=True)

# Display the combined recovery probability table
print("\nCombined Recovery Probability Table")
display(recovery_summary)

As we can see from the summarized table above, Google users recover the slowest and and Other users recover the fastest. Mozilla users are second to fastest.  

One thing to note, is that if a Other user has yet to recover by 6 hours, they won't for 3 days. Whereas the rest slowly still increase recovery overy time.

In [0]:
# Plot recovery summary by mobile brand name
plot_recovery_summary(recovery_summary, 'device_mobile_brand_name')

Clearly, Mozilla and Other have the highest rates of recovery. The rest are slightly lower, but still all have diminishing returns over time.

In [0]:
# Analyze recovery probability for each device operating system
windows = analyze_device(events_recovered, 'device_operating_system', 'Windows')
ios = analyze_device(events_recovered, 'device_operating_system', 'iOS')
macintosh = analyze_device(events_recovered, 'device_operating_system', 'Macintosh')
android = analyze_device(events_recovered, 'device_operating_system', 'Android')
chrome = analyze_device(events_recovered, 'device_operating_system', 'Chrome OS')

# Combine all device-specific results into a single summary table
recovery_summary = pd.concat([windows, ios, macintosh, android, chrome])

# Display the combined recovery probability summary
print("\nCombined Recovery Probability Table")
display(recovery_summary)

As we can see from the summarized table above, Windows users recover the slowest and and Android users recover the fastest. They all slowly increase recovery over time.

In [0]:
# Plot the recovery summary grouped by device operating system
plot_recovery_summary(recovery_summary, 'device_operating_system')

As we can see from the plot above, Android users clearly recover more in a shorter timeframe than the rest, who seem to have similar recovery rates over time.

## Q3 & Q4 Modeling

In [0]:
# Initialize random forest model
rf = RandomForestClassifier(
    random_state=seed,
    n_jobs=-1,
    max_samples=0.7,
    bootstrap=True
)

# Create model pipeline
rf_pipe = Pipeline([("prep", preprocessor), ("model", rf)])

# Define hyperparameter grid
rf_params = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [8, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", "log2"],
}

# Run randomized search CV
rf_search = RandomizedSearchCV(
    rf_pipe,
    rf_params,
    n_iter=3,
    cv=2,
    scoring="f1",
    n_jobs=-1,
    random_state=seed,
    verbose=2,
)

print("\nRF: quick CV on subset")
rf_search.fit(X_train_cv, y_train_cv)
print("RF best params (subset):", rf_search.best_params_)

# Refit best model on full training data
rf_best = rf_search.best_estimator_
rf_best.fit(X_train, y_train)

# Evaluate performance on test data
evaluate_on_test("Random Forest", rf_best)

We trained and tuned a Random Forest model to predict cart abandonment using a small randomized hyperparameter search, and the best configuration selected 150 trees with unrestricted depth, a slightly more conservative minimum split size, and log2 feature sampling to increase diversity among trees. This combination suggests the model benefits from flexibility to capture non linear patterns while still controlling overfitting. Random forest is excellent at modelling non linear data.

Evaluated on the test set, the model achieved an overall accuracy of 82.8%, which appears strong at first glance. However, because abandonment is a rare event in our dataset, accuracy alone is misleading. The confusion matrix reveals that the model performs very well at identifying customers who do not abandon (True Negative rate of 93.1 percent), but struggles to correctly detect customers who actually abandoned. For the minority class, precision was 27.3 percent and recall was 16.7 percent, resulting in an F1-score of 0.207. In other words, when the model predicts abandonment, it is only correct about one quarter of the time, and it currently only catches about one sixth of true abandoners. The low recall indicates many lost revenue opportunities are still slipping through undetected.

The macro average performance metrics (F1 = 0.555) more honestly reflect the imbalance in the data, while the weighted-average metrics are skewed by the dominant class. We also calculated the Precision-Recall AUC (0.205), which is an appropriate metric for rare event modeling. This value indicates the model performs moderately above random, but there is substantial room for improvement. Overall, this version of the Random Forest is useful as a ranking tool, as it can help us prioritize which customers are more likely to abandon relative to others, but it is not yet strong enough to serve as a direct trigger for retention interventions.

In [0]:
# Initialize the logistic regression model
log_reg = LogisticRegression(
    solver='saga',
    penalty='l2',
    C=1.0,
    class_weight='balanced',
    max_iter=500,
    tol=1e-3,
    warm_start=False,
    random_state=seed,
)

# Create a modeling pipeline without caching
log_pipe = Pipeline(
    steps=[
        ('prep', preprocessor),
        ('model', log_reg),
    ]
)

# Define a simple hyperparameter grid for regularization strength
log_params = {'model__C': [0.1, 0.3, 1.0]}

# Perform a randomized cross-validation search
log_search = RandomizedSearchCV(
    estimator=log_pipe,
    param_distributions=log_params,
    n_iter=3,
    cv=2,
    scoring='f1',
    n_jobs=1,
    random_state=seed,
    verbose=2,
    refit=True,
)

# Fit the randomized search on a CV subset
print('\nLogReg: quick CV on subset')
log_search.fit(X_train_cv, y_train_cv)
print('LogReg best params (subset):', log_search.best_params_)

# Rebuild the best model using the optimal hyperparameters
best_C = log_search.best_params_['model__C']
log_best = Pipeline(
    steps=[
        ('prep', preprocessor),
        (
            'model',
            LogisticRegression(
                solver='saga',
                penalty='l2',
                C=best_C,
                class_weight='balanced',
                max_iter=1000,
                tol=1e-3,
                warm_start=False,
                random_state=seed,
            ),
        ),
    ]
)

# Fit the final model on the full training dataset
print('\nRefitting best Logistic Regression on FULL train')
log_best.fit(X_train, y_train)

# Evaluate model performance on the test dataset
evaluate_on_test('Logistic Regression', log_best)

We trained and tuned a Logistic Regression model using a small randomized hyperparameter search focused on regularization strength. The best performing model used a relatively stronger regularization value (C = 0.1). Lower values of C apply more regularization, which helps prevent overfitting and encourages the model to rely on only the most consistently predictive signals. Because Logistic Regression is inherently linear, it tends to prefer simpler solutions, and the class_weight='balanced' parameter was included to compensate for the severe imbalance between abandoned and non abandoned cases.

When evaluated on the test set, this model achieved an overall accuracy of 52.8 percent. Although accuracy appears lower than tree based models, this drop is expected because Logistic Regression does not automatically learn complex non linear interactions. The confusion matrix shows that the model correctly identifies some abandoners that the Random Forest missed, but it also incorrectly flags a large number of customers who did not abandon. This trade off occurs because class balancing pushes the model to be more aggressive in predicting the minority class. As a result, precision and recall for the abandoned class are more balanced, but still modest, and the F1-score indicates that performance remains limited in this challenging rare-event setting.

From a fairness and ranking perspective, Logistic Regression offers clear coefficient-based interpretability, allowing us to understand which features most strongly increase or decrease abandonment risk. However, its linear decision boundary limits its ability to capture complex patterns in customer behavior and cart activity. The macro average metrics again illustrate the impact of class imbalance, while the weighted averages are skewed toward the dominant non abandonment class. The Precision Recall AUC score of 0.194 suggests slightly above random performance, but it underperforms the Random Forest's ability to isolate subtle abandonment behavior.

Overall, Logistic Regression provides a useful baseline model that is fast, interpretable, and simple, but it is not well suited to modeling the complex, non linear signals involved in cart abandonment.

In [0]:
# Initialize XGBoost model
xgb = XGBClassifier(
    random_state=seed,
    tree_method="hist",
    eval_metric="logloss",
    n_jobs=-1
)

# Create model pipeline
xgb_pipe = Pipeline([("prep", preprocessor), ("model", xgb)])

# Define hyperparameter grid
xgb_params = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [4, 6],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
}

# Run randomized search CV
xgb_search = RandomizedSearchCV(
    xgb_pipe,
    xgb_params,
    n_iter=3,
    cv=2,
    scoring="f1",
    n_jobs=-1,
    random_state=seed,
    verbose=2,
)

print("\nXGB: quick CV on subset")
xgb_search.fit(X_train_cv, y_train_cv)
print("XGB best params (subset):", xgb_search.best_params_)

# Refit best model on full training data
xgb_best = xgb_search.best_estimator_
xgb_best.fit(X_train, y_train)

# Evaluate performance on test data
evaluate_on_test("XGBoost", xgb_best)

We trained and tuned an XGBoost classification model using a small randomized hyperparameter search that explored the number of boosting rounds, tree depth, learning rate, and sampling strategies. The best performing configuration used 250 trees, a relatively shallow max depth of 4, and a learning rate of 0.1. This setup suggests that the model benefits from taking a larger number of smaller, incremental steps while keeping individual trees simple. The subsample rate of 0.8 and full column sampling (colsample_bytree = 1.0) help introduce controlled randomness, which reduces overfitting and increases generalization.

When evaluated on the test set, XGBoost achieved an overall accuracy of 83.0 percent. Similar to the Random Forest, this high accuracy is mostly driven by strong performance on the dominant non abandonment carts. The confusion matrix shows a True Negative rate of 93.4 percent, indicating the model is very reliable at identifying customers who will not abandon. However, recall for the abandoned class remains relatively low at 16.0 percent, meaning the model is only capturing a small portion of customers who actually abandon their carts. Precision for this class is also modest at 27.4 percent, so only about one quarter of predicted abandoners truly abandon. Together, these scores produce a minority class F1-score of 0.202, which reflects the difficulty of detecting rare abandonment behavior.

The macro averaged metrics (F1 = 0.554) better represent the imbalance in the outcome variable, whereas the weighted average is skewed by the large volume of non abandonment. The Precision Recall AUC score of 0.2042 suggests performance moderately above random, and it nearly matches the Random Forest, indicating that both tree-based approaches are capturing similar signal from our features.

Overall, XGBoost provides competitive performance, efficiently models non linear relationships, and benefits from boosting based refinement that sequentially corrects prior errors. However, like the other models, it still struggles to identify abandoners due to severe class imbalance and limited signal in the event data.

In [0]:
# Prepare dataframe and build features
post_eventns_x = post_eventns.copy()
post_eventns_x["event_date"] = pd.to_datetime(
    post_eventns_x["event_date"], errors="coerce"
)
post_eventns_x["event_ts_utc"] = pd.to_datetime(
    post_eventns_x["event_ts_utc"], errors="coerce"
)
post_eventns_x["dow"] = post_eventns_x["event_date"].dt.dayofweek
post_eventns_x["hour"] = post_eventns_x["event_ts_utc"].dt.hour
post_eventns_x = post_eventns_x.dropna().reset_index(drop=True)

# Define features and target
y = post_eventns_x["abandoned"].astype(int)
feat_cols = [
    "event_name",
    "device_category",
    "device_mobile_brand_name",
    "device_operating_system",
    "event_page_name",
    "quantity",
    "dow",
    "hour",
]
X = post_eventns_x[feat_cols]

# Split into train and test sets
X_tr_all, X_test, y_tr_all, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=seed
)

# Create validation split from training data
X_tr_raw, X_val_raw, y_tr_raw, y_val_raw = train_test_split(
    X_tr_all, y_tr_all, test_size=0.10, stratify=y_tr_all, random_state=seed
)

# Build downsampled train variants
train_variants = {
    "ds_2to1": downsample(X_tr_raw, y_tr_raw, ratio=2),
    "ds_1to1": downsample(X_tr_raw, y_tr_raw, ratio=1),
}

# Run model tuning and evaluation with sklearn XGBClassifier
configs = []
param_grid = [
    {
        "eta": 0.05,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "spw": 1.0,
    },
    {
        "eta": 0.05,
        "max_depth": 8,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "spw": 1.2,
    },
    {
        "eta": 0.10,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "spw": 1.0,
    },
    {
        "eta": 0.10,
        "max_depth": 6,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "spw": 1.5,
    },
    {
        "eta": 0.05,
        "max_depth": 6,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "spw": 1.5,
    },
    {
        "eta": 0.10,
        "max_depth": 4,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "spw": 2.0,
    },
]

# Train and evaluate across parameter grid
for tag, (Xtr, ytr) in train_variants.items():
    pre, num_cols, cat_cols = build_preprocessor(Xtr)
    Xtr_p = pre.fit_transform(Xtr)
    Xval_p = pre.transform(X_val_raw)
    Xtest_p = pre.transform(X_test)

    neg = int((ytr == 0).sum())
    pos = int((ytr == 1).sum())
    base_spw = max(1.0, neg / max(1, pos))

    for p in param_grid:
        clf = XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            eval_metric="logloss",
            n_estimators=800,
            learning_rate=p["eta"],
            max_depth=p["max_depth"],
            subsample=p["subsample"],
            colsample_bytree=p["colsample_bytree"],
            scale_pos_weight=base_spw * p["spw"],
            random_state=seed,
            n_jobs=-1,
        )

        # Fit the model using validation set; omit early_stopping_rounds for compatibility
        clf.fit(
            Xtr_p,
            ytr,
            eval_set=[(Xval_p, y_val_raw)],
            verbose=False,
        )

        best_iter = getattr(clf, "best_iteration_", None)

        yv = clf.predict_proba(Xval_p)[:, 1]
        prec, rec, thr = precision_recall_curve(y_val_raw, yv)
        f1 = 2 * (prec * rec) / (prec + rec + 1e-12)
        best_idx = int(np.nanargmax(f1))
        thr_star = thr[min(best_idx, len(thr) - 1)] if len(thr) > 0 else 0.5
        val_f1 = float(f1[best_idx])
        val_ap = float(average_precision_score(y_val_raw, yv))

        configs.append(
            {
                "variant": tag,
                "params": p,
                "best_iter": best_iter,
                "val_f1": val_f1,
                "val_ap": val_ap,
                "thr": float(thr_star),
                "clf": clf,
                "pre": pre,
                "Xtest_p": Xtest_p,
            }
        )

# Rank and display leaderboard
configs = sorted(configs, key=lambda z: z["val_f1"], reverse=True)
print("\nXGB Leaderboard (by Validation F1 for class=1)")
for r in configs[:5]:
    print(
        f"{r['variant']} | F1={r['val_f1']:.3f} | PR-AUC={r['val_ap']:.3f} "
        f"| thr={r['thr']:.3f} | iter={r['best_iter']} | params={r['params']}"
    )

# Evaluate best model on test data
best = configs[0]
pre = best["pre"]
Xtest_p = best["Xtest_p"]
y_test_prob = best["clf"].predict_proba(Xtest_p)[:, 1]
y_test_pred = (y_test_prob >= best["thr"]).astype(int)

print("\nTEST at tuned threshold (picked on VAL F1)")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred, digits=3))
print(f"Test PR-AUC: {average_precision_score(y_test, y_test_prob):.4f}")
print("\nBest config used:")
print(
    best["variant"],
    best["params"],
    "best_iter:",
    best["best_iter"],
    "thr:",
    round(best["thr"], 3),
)

In this test, we trained multiple XGBoost models using different hyperparameter configurations and applied downsampling to partially rebalance the dataset before fitting. Because abandonment is rare, we also tuned the scale_pos_weight parameter, which increases the loss penalty when the model misclassifies an abandoned cart. We evaluated all configurations on a held out validation set and selected the model that maximized the F1 score for the abandoned class. Unlike earlier experiments, we then set the final decision threshold using the point on the validation curve that provided the best precision recall trade-off for abandoners, rather than relying on the default 0.50 cutoff.

On the test set, this tuned XGBoost model demonstrated much higher recall for abandoned carts (68.2 percent), meaning it successfully identifies a much larger portion of customers who will ultimately abandon. However, this improvement comes at the cost of substantially lower precision (17.9 percent), indicating many false positives. In practice, this means the model is aggressive. it catches far more true abandoners but also incorrectly flags many customers who will end up purchasing. Overall accuracy drops to 53.6 percent, which is expected because we intentionally shifted the threshold to favor minority-class recall.

The Test PR-AUC of 0.2053 closely matches our earlier boosting results, suggesting that even with threshold tuning and downsampling, abandonment remains difficult to predict from event level features alone. This configuration is useful if the business goal is to maximize saves and is willing to tolerate more outreach to non-abandoners. However, additional behavioral signals, cart context, or historical customer profiles would likely be required to raise both precision and recall simultaneously.

In [0]:
# Calculate abandonment rate by device category
device_abandon = (
    post_eventns.groupby("device_category")["abandoned"]
    .mean()
    .reset_index()
    .rename(columns={"abandoned": "abandonment_rate"})
    .sort_values("abandonment_rate", ascending=False)
)

plt.figure(figsize=(7, 4))
sns.barplot(
    x="device_category",
    y="abandonment_rate",
    data=device_abandon,
    palette="crest"
)
plt.title("Cart Abandonment Rate by Device Type")
plt.ylabel("Abandonment Rate (%)")
plt.show()

Out of all three device types, tablets have the highest abandonment rate followed by mobile then desktop.

In [0]:
# Filter data to only abandoned records
abandoned_post_eventns = post_eventns[post_eventns["abandoned"] == True].copy()

# Count abandoned item IDs
item_counts = (
    abandoned_post_eventns["item_id"]
    .astype(str)
    .value_counts()
    .to_frame("count")
    .reset_index()
    .rename(columns={"index": "item_id"})
)

# Identify product name column in materials
name_col = None
for candidate in ["trademark", "trade_mark_desc", "product_name", "material_desc", "name"]:
    if candidate in materials.columns:
        name_col = candidate
        break

if name_col is None:
    raise KeyError(
        "Couldn't find a product name/description column in `materials`. "
        "Tried: ['trademark','trade_mark_desc','product_name','material_desc','name'].\n"
        f"materials columns are: {list(materials.columns)}"
    )

# Ensure merge key data type alignment
materials = materials.copy()
if "material_id" not in materials.columns:
    raise KeyError("`materials` is missing 'material_id' column.")
materials["material_id"] = materials["material_id"].astype(str)

# Merge item counts with materials data
merged = item_counts.merge(
    materials[["material_id", name_col]],
    left_on="item_id",
    right_on="material_id",
    how="left"
)

# Combine counts by product or trademark
grouped = (
    merged.groupby(name_col, dropna=False)["count"]
    .sum()
    .reset_index()
    .rename(columns={name_col: "product_name"})
    .sort_values("count", ascending=False)
)

# Select top 10 abandoned products
top10_products = grouped.head(10).reset_index(drop=True)

# Plot top abandoned products
plt.figure(figsize=(9, 5))
labels = top10_products["product_name"].fillna("(unknown)")
counts = top10_products["count"]

plt.barh(labels[::-1], counts[::-1])
plt.xlabel("Number of Abandoned Carts")
plt.ylabel("Product (Trademark)")
plt.title("Top 10 Most Frequently Abandoned Products")
plt.tight_layout()
plt.show()

The top 3 most frequently abandoned products are: Fizz Factory, Oliver Originals and Petes Popcorn.

## Q5 Modeling

In [0]:
fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Plot realized and potential revenue
axs[0].plot(daily["event_date"], daily["realized_revenue"], label="Realized Revenue")
axs[0].plot(daily["event_date"], daily["potential_revenue"], label="Potential Revenue")

# Add trend line for realized revenue
z_realized = np.polyfit(daily.index, daily["realized_revenue"], 1)
trend_realized = np.polyval(z_realized, daily.index)
axs[0].plot(daily["event_date"], trend_realized, color="tab:blue", linestyle="--", label="Realized Trend")

# Add trend line for potential revenue
z_potential = np.polyfit(daily.index, daily["potential_revenue"], 1)
trend_potential = np.polyval(z_potential, daily.index)
axs[0].plot(daily["event_date"], trend_potential, color="tab:orange", linestyle="--", label="Potential Trend")

# Configure revenue plot
axs[0].set_title("Daily Revenue: Realized vs Potential")
axs[0].set_ylabel("Revenue")
axs[0].legend()

# Plot abandonment rate
axs[1].plot(daily["event_date"], daily["abandon_rate"], color="tab:orange")

# Add trend line for abandonment rate
z_abandon = np.polyfit(daily.index, daily["abandon_rate"], 1)
trend_abandon = np.polyval(z_abandon, daily.index)
axs[1].plot(daily["event_date"], trend_abandon, color="tab:red", linestyle="--", label="Abandonment Trend")

# Configure abandonment plot
axs[1].set_title("Daily Abandonment Rate")
axs[1].set_xlabel("Date")
axs[1].set_ylabel("Abandonment Rate")
axs[1].legend()

plt.tight_layout()
plt.show()


# Aggregate weekly revenue and abandonment data
weekly = (
    daily.set_index("event_date")[["realized_revenue", "potential_revenue", "abandon_rate"]]
    .resample("W")
    .mean()
)
weekly = weekly.reset_index()

fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Plot realized and potential revenue (weekly)
axs[0].plot(weekly["event_date"], weekly["realized_revenue"], label="Realized Revenue")
axs[0].plot(weekly["event_date"], weekly["potential_revenue"], label="Potential Revenue")

# Weekly trend line for realized revenue
z_realized_w = np.polyfit(np.arange(len(weekly)), weekly["realized_revenue"], 1)
trend_realized_w = np.polyval(z_realized_w, np.arange(len(weekly)))
axs[0].plot(weekly["event_date"], trend_realized_w, color="tab:blue", linestyle="--", label="Realized Trend")

# Weekly trend line for potential revenue
z_potential_w = np.polyfit(np.arange(len(weekly)), weekly["potential_revenue"], 1)
trend_potential_w = np.polyval(z_potential_w, np.arange(len(weekly)))
axs[0].plot(weekly["event_date"], trend_potential_w, color="tab:orange", linestyle="--", label="Potential Trend")

# Configure weekly revenue plot
axs[0].set_title("Weekly Revenue: Realized vs Potential")
axs[0].set_ylabel("Revenue")
axs[0].legend()

# Plot weekly abandonment rate
axs[1].plot(weekly["event_date"], weekly["abandon_rate"], color="tab:orange")

# Weekly trend line for abandonment rate
z_abandon_w = np.polyfit(np.arange(len(weekly)), weekly["abandon_rate"], 1)
trend_abandon_w = np.polyval(z_abandon_w, np.arange(len(weekly)))
axs[1].plot(weekly["event_date"], trend_abandon_w, color="tab:red", linestyle="--", label="Abandonment Trend")

# Configure weekly abandonment plot
axs[1].set_title("Weekly Abandonment Rate")
axs[1].set_xlabel("Week")
axs[1].set_ylabel("Abandonment Rate")
axs[1].legend()

plt.tight_layout()
plt.show()


In [0]:
# # Quick exploratory visuals for revenue and abandonment trends
# fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# # Plot realized and potential revenue
# axs[0].plot(daily["event_date"], daily["realized_revenue"], label="Realized Revenue")
# axs[0].plot(daily["event_date"], daily["potential_revenue"], label="Potential Revenue")

# # Add trend line for realized revenue
# z_realized = np.polyfit(daily.index, daily["realized_revenue"], 1)
# trend_realized = np.polyval(z_realized, daily.index)
# axs[0].plot(daily["event_date"], trend_realized, color="tab:blue", linestyle="--", label="Realized Trend")

# # Add trend line for potential revenue
# z_potential = np.polyfit(daily.index, daily["potential_revenue"], 1)
# trend_potential = np.polyval(z_potential, daily.index)
# axs[0].plot(daily["event_date"], trend_potential, color="tab:orange", linestyle="--", label="Potential Trend")

# # Configure revenue plot
# axs[0].set_title("Daily Revenue: Realized vs Potential")
# axs[0].set_ylabel("Revenue")
# axs[0].legend()

# # Plot abandonment rate
# axs[1].plot(daily["event_date"], daily["abandon_rate"], color="tab:orange")

# # Add trend line for abandonment rate
# z_abandon = np.polyfit(daily.index, daily["abandon_rate"], 1)
# trend_abandon = np.polyval(z_abandon, daily.index)
# axs[1].plot(daily["event_date"], trend_abandon, color="tab:red", linestyle="--", label="Abandonment Trend")

# # Configure abandonment plot
# axs[1].set_title("Daily Abandonment Rate")
# axs[1].set_xlabel("Date")
# axs[1].set_ylabel("Abandonment Rate")
# axs[1].legend()

# # Render final layout
# plt.tight_layout()
# plt.show()

Our initial visuals show clear differences between potential and realized revenue over time, highlighting the financial impact of cart abandonment on MyCoke360. While both potential and realized revenue demonstrate steady growth, the consistent gap between the two indicates recurring revenue loss from uncompleted transactions. The positive trend in potential revenue suggests increasing customer engagement and order activity, but realized revenue lags slightly behind, reflecting that a portion of those purchase opportunities are not being converted.

The lower chart shows that abandonment rates fluctuate considerably but exhibit a mild upward trend, implying that as traffic and potential sales increase, abandonment behavior is not decreasing proportionally. This persistent abandonment trend contributes to a widening gap between potential and realized revenue, reducing overall profitability and likely affecting the mix of products actually sold. High-margin or frequently abandoned product categories may therefore be disproportionately underrepresented in realized sales, amplifying the financial effect on total revenue performance.

In [0]:
# Prepare time series for modeling
ts = daily[["event_date", "realized_revenue"]].dropna().copy()
ts = ts.set_index("event_date").asfreq("D").fillna(0.0)

# Split data into training and validation sets
split_idx = int(len(ts) * 0.8)
y_train = ts.iloc[:split_idx]["realized_revenue"]
y_valid = ts.iloc[split_idx:]["realized_revenue"]

# Define SARIMA parameter grids for non-seasonal and seasonal components
pdq_grid = [(p, d, q) for p in [0, 1, 2] for d in [0, 1] for q in [0, 1, 2]]
seasonal_grid = [(P, D, Q, 7) for P in [0, 1] for D in [0, 1] for Q in [0, 1]]

# Initialize tracking variables for best configuration
best_cfg = None
best_aic = np.inf

# Perform grid search over SARIMA configurations
for (p, d, q) in pdq_grid:
    for (P, D, Q, s) in seasonal_grid:
        try:
            model = SARIMAX(
                y_train,
                order=(p, d, q),
                seasonal_order=(P, D, Q, s),
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            res = model.fit(disp=0)
            if res.aic < best_aic:
                best_aic = res.aic
                best_cfg = ((p, d, q), (P, D, Q, s))
        except Exception:
            continue

# Display best model configuration
print(f"Best SARIMA cfg: {best_cfg}  AIC={best_aic:.1f}")

# Fit the best SARIMA model
order, seasonal_order = best_cfg
final_model = SARIMAX(
    y_train,
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=0)

# Generate forecasts for the validation horizon
fc = final_model.get_forecast(steps=len(y_valid)).predicted_mean

# Evaluate model performance with RMSE
rmse = mean_squared_error(y_valid, fc, squared=False)
print(f"Validation RMSE: {rmse:,.2f}")

# Create plots for model evaluation
fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Plot SARIMA forecast for realized revenue
axs[0].plot(y_train.index, y_train.values, label="Train")
axs[0].plot(y_valid.index, y_valid.values, label="Actual")
axs[0].plot(y_valid.index, fc.values, label="Forecast")
axs[0].set_title("SARIMA Forecast: Realized Revenue")
axs[0].set_ylabel("Revenue")
axs[0].legend()

# Plot daily lost revenue due to abandonment
axs[1].plot(daily["event_date"], daily["abandoned_revenue"], color="tab:orange")
axs[1].set_title("Daily Lost Revenue (Abandonment)")
axs[1].set_xlabel("Date")
axs[1].set_ylabel("Lost Revenue")

# Finalize and display plots
plt.tight_layout()
plt.show()

The results in the figure show both the historical and forecasted effects of cart abandonment on MyCoke360's daily revenue. The top panel presents a SARIMA forecast of realized revenue that distinguishes between the training, actual, and predicted periods. The model captures consistent seasonality and volatility in daily sales, with the forecast indicating continued fluctuations within the observed historical range. This suggests that the revenue pattern is cyclical and stable, reflecting regular customer purchasing behavior on the platform.

The lower panel displays daily lost revenue resulting from cart abandonment. It shows that revenue loss is consistent and has increased over time, with peaks aligning with periods of higher sales activity. This pattern indicates that as overall sales volume grows, so does the financial impact of cart abandonment. In recent periods, lost revenue reached notable levels, meaning even small improvements in conversion rates could produce meaningful revenue recovery.

Overall, the analysis demonstrates that cart abandonment significantly reduces MyCoke360's realized revenue relative to its potential revenue and influences the product mix by limiting completed sales of certain items. The forecast implies that without changes to customer engagement or checkout processes, the platform will continue to experience predictable and recurring losses tied to abandoned transactions.

In [0]:
# In-sample residual diagnostics
_ = final_model.plot_diagnostics(figsize=(12, 8))

# Prepare in-sample residuals
resid_in = final_model.resid.dropna()

# Prepare fitted values aligned to residual index
fitted_in = final_model.fittedvalues.reindex(resid_in.index)

# Overlay actual vs fitted on training data
plt.figure(figsize=(12, 4))
plt.plot(y_train.index, y_train.values, label="Train actual")
plt.plot(fitted_in.index, fitted_in.values, label="Train fitted")
plt.title("Train: actual vs fitted")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.tight_layout()
plt.show()

The above diagnostic plots provide a strong indication that the SARIMA model is performing reasonably well in capturing the overall trend and seasonality of MyCoke360's realized revenue. The standardized residuals fluctuate closely around zero without displaying a consistent pattern, suggesting that the model has successfully removed systematic variation from the data. The histogram and kernel density estimates show that the residuals are approximately normally distributed, though there is a slight skew and heavier tails, implying some deviation from perfect normality that may stem from high volatility in daily sales. The Q-Q plot supports this observation, as most points align along the reference line except for a few outliers in the extremes, which are typical in financial time series where sporadic spikes occur. The correlogram shows no significant autocorrelation beyond lag zero, confirming that the residuals are largely uncorrelated and that the model has captured most of the temporal dependencies in the series.

The training fit plot demonstrates that the model closely follows the actual revenue pattern across time, reproducing both the amplitude and frequency of recurring revenue cycles. Minor underestimations and overestimations are visible during sharp peaks and troughs, which indicates that the model slightly lags in reacting to sudden changes in customer purchasing behavior. Despite these localized discrepancies, the model maintains a strong correspondence between predicted and actual values, reflecting a high level of explanatory power for the observed revenue dynamics.

Overall, the model diagnostics indicate that the SARIMA model provides a reliable basis for forecasting MyCoke360's realized revenue and for evaluating the financial effect of cart abandonment. The model captures the dominant structure of the data and produces largely unbiased residuals, which supports confidence in the conclusions drawn from its forecasts. While further refinement could improve handling of extreme revenue spikes, the model's performance is sufficiently accurate to justify the earlier interpretations regarding revenue impact and abandonment-driven loss patterns.

In [0]:
# Prophet model for forecasting realized revenue
prophet_df = daily[["event_date", "realized_revenue"]].rename(
    columns={"event_date": "ds", "realized_revenue": "y"}
)
prophet_df = prophet_df.sort_values("ds")
prophet_df["ds"] = pd.to_datetime(prophet_df["ds"]).dt.tz_localize(None)

# Split data into training and validation sets (80/20)
cut_idx = int(len(prophet_df) * 0.8)
train_p = prophet_df.iloc[:cut_idx].copy()
valid_p = prophet_df.iloc[cut_idx:].copy()

# Define parameter grid for changepoint and seasonality tuning
param_grid = [
    {"changepoint_prior_scale": 0.05, "seasonality_mode": "additive"},
    {"changepoint_prior_scale": 0.5, "seasonality_mode": "additive"},
    {"changepoint_prior_scale": 0.05, "seasonality_mode": "multiplicative"},
    {"changepoint_prior_scale": 0.5, "seasonality_mode": "multiplicative"},
]

# Initialize tracking variables for best results
best_rmse = np.inf
best_params = None
best_valid = None

# Iterate through parameter combinations
for params in param_grid:
    m = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=True,
        changepoint_prior_scale=params["changepoint_prior_scale"],
        seasonality_mode=params["seasonality_mode"],
    )
    m.fit(train_p)
    future = m.make_future_dataframe(periods=len(valid_p), freq="D")
    fcst = m.predict(future)
    pred = fcst.tail(len(valid_p))["yhat"].values
    rmse_p = mean_squared_error(valid_p["y"].values, pred, squared=False)

    # Store best model parameters and results
    if rmse_p < best_rmse:
        best_rmse = rmse_p
        best_params = params
        best_valid = (valid_p["ds"], valid_p["y"].values, pred)

# Output best parameters and RMSE
print(f"Best Prophet params: {best_params}  RMSE={best_rmse:,.2f}")

# Daily validation visual
ds_valid, y_true, y_hat = best_valid
plt.figure(figsize=(10, 4))
plt.plot(train_p["ds"], train_p["y"], label="Train")
plt.plot(ds_valid, y_true, label="Actual")
plt.plot(ds_valid, y_hat, label="Forecast")
plt.title("Prophet Forecast: Realized Revenue (Daily)")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.tight_layout()
plt.show()

# Weekly aggregation visual
# Prepare combined daily data with segment labels
train_plot = train_p[["ds", "y"]].copy()
train_plot["segment"] = "Train"

valid_actual_plot = pd.DataFrame(
    {"ds": ds_valid, "y": y_true, "segment": "Actual"}
)
valid_fc_plot = pd.DataFrame(
    {"ds": ds_valid, "y": y_hat, "segment": "Forecast"}
)

combined = pd.concat(
    [train_plot, valid_actual_plot, valid_fc_plot],
    ignore_index=True,
)

# Aggregate to weekly sums
weekly_agg = (
    combined.set_index("ds")
    .groupby("segment")["y"]
    .resample("W")
    .sum()
    .reset_index()
)

# Pivot for plotting
weekly_pivot = weekly_agg.pivot(index="ds", columns="segment", values="y")

plt.figure(figsize=(10, 4))
if "Train" in weekly_pivot.columns:
    plt.plot(weekly_pivot.index, weekly_pivot["Train"], label="Train")
if "Actual" in weekly_pivot.columns:
    plt.plot(weekly_pivot.index, weekly_pivot["Actual"], label="Actual")
if "Forecast" in weekly_pivot.columns:
    plt.plot(weekly_pivot.index, weekly_pivot["Forecast"], label="Forecast")

plt.title("Prophet Forecast: Realized Revenue (Weekly Aggregation)")
plt.xlabel("Week")
plt.ylabel("Revenue")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
# Prophet model for forecasting realized revenue
# prophet_df = daily[["event_date", "realized_revenue"]].rename(
#     columns={"event_date": "ds", "realized_revenue": "y"}
# )
# prophet_df = prophet_df.sort_values("ds")
# prophet_df["ds"] = pd.to_datetime(prophet_df["ds"]).dt.tz_localize(None)

# # Split data into training and validation sets (80/20)
# cut_idx = int(len(prophet_df) * 0.8)
# train_p = prophet_df.iloc[:cut_idx].copy()
# valid_p = prophet_df.iloc[cut_idx:].copy()

# # Define parameter grid for changepoint and seasonality tuning
# param_grid = [
#     {"changepoint_prior_scale": 0.05, "seasonality_mode": "additive"},
#     {"changepoint_prior_scale": 0.5, "seasonality_mode": "additive"},
#     {"changepoint_prior_scale": 0.05, "seasonality_mode": "multiplicative"},
#     {"changepoint_prior_scale": 0.5, "seasonality_mode": "multiplicative"},
# ]

# # Initialize tracking variables for best results
# best_rmse = np.inf
# best_params = None
# best_valid = None

# # Iterate through parameter combinations
# for params in param_grid:
#     m = Prophet(
#         weekly_seasonality=True,
#         yearly_seasonality=True,
#         changepoint_prior_scale=params["changepoint_prior_scale"],
#         seasonality_mode=params["seasonality_mode"],
#     )
#     m.fit(train_p)
#     future = m.make_future_dataframe(periods=len(valid_p), freq="D")
#     fcst = m.predict(future)
#     pred = fcst.tail(len(valid_p))["yhat"].values
#     rmse_p = mean_squared_error(valid_p["y"].values, pred, squared=False)

#     # Store best model parameters and results
#     if rmse_p < best_rmse:
#         best_rmse = rmse_p
#         best_params = params
#         best_valid = (valid_p["ds"], valid_p["y"].values, pred)

# # Output best parameters and RMSE
# print(f"Best Prophet params: {best_params}  RMSE={best_rmse:,.2f}")

# # Plot validation results for the best model
# ds_valid, y_true, y_hat = best_valid
# plt.figure(figsize=(10, 4))
# plt.plot(train_p["ds"], train_p["y"], label="Train")
# plt.plot(ds_valid, y_true, label="Actual")
# plt.plot(ds_valid, y_hat, label="Forecast")
# plt.title("Prophet Forecast: Realized Revenue")
# plt.xlabel("Date")
# plt.ylabel("Revenue")
# plt.legend()
# plt.show()

The Prophet forecast illustrates MyCoke360's realized revenue patterns with close alignment between the actual and predicted data. The model captures strong recurring peaks and troughs that reflect regular sales cycles and customer purchasing behavior. Prophet's ability to adapt to short-term fluctuations and trend shifts allows it to respond more quickly to changing revenue conditions than the SARIMA model, which produced smoother but less reactive forecasts. This flexibility provides a clearer view of the timing and intensity of sales variations that may be linked to cart abandonment or seasonal promotions.

Compared to SARIMA, Prophet shows greater short-term sensitivity and a slightly wider range of predicted values. This indicates that Prophet is better at identifying emerging changes in customer activity and the immediate revenue effects of abandonment, while SARIMA remains more stable for long-term trend estimation. Prophet's higher variance suggests that it captures the natural volatility of e-commerce sales but may be less reliable for extended forecasting horizons.

In evaluating the financial impact of cart abandonment, the Prophet model confirms that revenue losses are both meaningful and dynamic. The close tracking of realized revenue shows that periods of higher customer activity lead to larger fluctuations in realized versus potential sales. This supports the interpretation that cart abandonment significantly reduces short-term revenue and alters the product mix by limiting conversions on higher-value products during busy sales periods. The model suggests that timely interventions focused on recovery and checkout optimization could produce measurable gains in overall revenue performance.

In [0]:
# Refit model using best parameters
m_best = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=best_params["changepoint_prior_scale"],
    seasonality_mode=best_params["seasonality_mode"],
)
m_best.fit(train_p)

# Compute fitted values on training data
fitted_train = (
    m_best.predict(train_p)[["ds", "yhat"]]
    .rename(columns={"yhat": "yhat_train"})
)
train_with_fit = train_p.merge(fitted_train, on="ds", how="left")

# Compute training residuals
resid_in = (train_with_fit["y"] - train_with_fit["yhat_train"]).dropna()

# Plot residuals versus fitted values (key diagnostic)
plt.figure(figsize=(6, 4))
plt.scatter(
    train_with_fit.loc[resid_in.index, "yhat_train"].values,
    resid_in.values,
    s=10,
)
plt.title("Prophet (train) residuals vs fitted")
plt.xlabel("Fitted (yhat)")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()

# Build unified dataframe of dates for component plots
try:
    future_all = pd.concat(
        [train_p[["ds"]], pd.DataFrame({"ds": ds_valid})],
        ignore_index=True,
    ).drop_duplicates()
except NameError:
    future_all = train_p[["ds"]].copy()

# Generate component plots (trend and seasonality)
fcst_all = m_best.predict(future_all)
_ = m_best.plot_components(fcst_all)
plt.tight_layout()
plt.show()


The diagnostic plots indicate that the Prophet model provides a credible representation of MyCoke360's realized revenue patterns, though it shows some systematic bias and variability in error magnitude. The residuals versus fitted plot reveals a generally random distribution around zero, which suggests that the model captures most of the predictable variation in revenue. However, there is a noticeable funnel shape in the residual spread, meaning prediction errors increase as revenue levels rise. This indicates that the model tends to underestimate or overestimate during extreme sales periods, a common limitation in revenue forecasting where outliers and large transaction spikes occur.

The trend plot shows a steady upward trajectory in overall revenue across the observed period, confirming consistent sales growth on the platform. The weekly component reveals clear cyclical variation, with the strongest positive effects concentrated early in the week and a decline toward weekends. This pattern implies that customer engagement and purchase activity are highest at the beginning of the week, possibly reflecting replenishment behavior or business order cycles. The yearly component shows distinct peaks and troughs corresponding to seasonal effects, such as promotional periods or product launches, suggesting that sales performance follows recurring annual demand cycles.

Taken together, these diagnostics show that the Prophet model effectively captures both long-term growth and recurring seasonality, producing a realistic depiction of revenue drivers. The residual structure indicates that while the model tracks average trends well, it may miss the full amplitude of revenue volatility during high-demand periods, where cart abandonment and conversion variability are also likely to increase. Despite this limitation, the model's general accuracy and clear representation of temporal patterns support the earlier interpretation that cart abandonment has a substantial and dynamic impact on MyCoke360's revenue and product mix. The consistent upward trend and recurring cycles reinforce that mitigating abandonment during peak demand periods would yield the most meaningful improvements in realized revenue.

In [0]:
# Set the top-k threshold for categorical bucketing
top_k = 15

# Create a working copy and apply category bucketing
# item_activity_copy = item_activity_data.copy()
for col in ["trademark", "flavor", "beverage_category", "packaging_type",
            "packaging_size"]:
    if col in item_activity_copy.columns:
        item_activity_copy[col] = top_k_bucket(item_activity_copy[col].astype(str), top_k)

# Define target and feature matrices
y = item_activity_copy["abandoned"].astype(int)
X = item_activity_copy[
    [
        "effective_unit_price",
        "effective_unit_cost",
        "effective_unit_profit",
        "trademark",
        "flavor",
        "beverage_category",
        "packaging_type",
        "packaging_size",
    ]
].copy()

# Specify column groups
num_cols = [
    "effective_unit_price",
    "effective_unit_cost",
    "effective_unit_profit",
]
cat_cols = ["trademark", "flavor", "beverage_category", "packaging_type",
            "packaging_size"]

# Build sparse-safe preprocessing
pre = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True),
         cat_cols),
    ],
    sparse_threshold=1.0,
)

# Configure logistic regression with L2 regularization and SAGA solver
logreg = LogisticRegression(
    solver="saga",
    penalty="l2",
    class_weight="balanced",
    max_iter=2000,
    tol=1e-3,
    random_state=42,
)

# Enable pipeline caching to avoid refitting encoders across folds
memory = Memory(location=Path("./.sk_cache"), verbose=0)

# Assemble the modeling pipeline
pipe = Pipeline(steps=[("pre", pre), ("clf", logreg)], memory=memory)

# Create train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Define randomized search space for C (log-uniform prior)
param_dist = {
    "clf__C": loguniform(1e-2, 1e2),
    # To extend search, you can add: "clf__penalty": ["l1", "l2"] with solver="saga"
}

# Run randomized search with AUC optimization
cv = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    random_state=42,
)
cv.fit(X_train, y_train)

# Evaluate the best model on the test set
best_lr = cv.best_estimator_
y_prob = best_lr.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)

# Plot the ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("Logistic Regression ROC")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.tight_layout()
plt.show()

# Print best parameters and test AUC
print(f"Best params: {cv.best_params_}")
print(f"Test AUC: {auc:.3f}")

The logistic regression ROC plot shows an AUC of 0.636, indicating that the model has modest ability to distinguish between abandoned and completed carts. The curve sits above the random baseline, meaning the model captures some meaningful patterns in customer behavior, but it is not highly predictive. This suggests that while logistic regression can identify general drivers of revenue loss due to cart abandonment, it may not be sufficient for accurate forecasting or precise customer targeting. The model offers a useful foundation for understanding financial impact trends, but additional features or more complex models are likely needed to improve performance and better quantify the true revenue at risk.

In [0]:
# Prepare an analysis-ready copy of the dataset
item_financials = item_activity_data.copy()

# Ensure core types are correct
item_financials["event_ts_utc"] = pd.to_datetime(
    item_financials["event_ts_utc"], utc=True, errors="coerce"
)
item_financials["abandoned"] = item_financials["abandoned"].astype(bool)
for _c in ["effective_unit_price", "effective_unit_profit", "effective_unit_cost"]:
    item_financials[_c] = pd.to_numeric(item_financials[_c], errors="coerce")

# Bucket high-cardinality product attributes to top-k plus Other
top_k = 15
for col in [
    "trademark",
    "flavor",
    "beverage_category",
    "packaging_type",
    "packaging_size",
]:
    if col in item_financials.columns:
        vc = item_financials[col].astype(str).value_counts(dropna=False)
        keep = set(vc.index[:top_k])
        item_financials[col] = item_financials[col].astype(str).where(
            item_financials[col].astype(str).isin(keep), other="Other"
        )

# Build the feature matrix expected by the trained pipeline
X = item_financials[
    [
        "effective_unit_price",
        "effective_unit_cost",
        "effective_unit_profit",
        "trademark",
        "flavor",
        "beverage_category",
        "packaging_type",
        "packaging_size",
    ]
].copy()

# Score abandonment probabilities with the trained model pipeline
p_abandon = best_lr.predict_proba(X)[:, 1]
item_financials["p_abandon"] = p_abandon

# Compute realized, potential, and expected lost revenue and profit
item_financials["potential_revenue"] = item_financials["effective_unit_price"]
item_financials["realized_revenue"] = np.where(
    item_financials["abandoned"], 0.0, item_financials["effective_unit_price"]
)
item_financials["expected_lost_revenue"] = (
    item_financials["p_abandon"] * item_financials["effective_unit_price"]
)

item_financials["potential_profit"] = item_financials["effective_unit_profit"]
item_financials["realized_profit"] = np.where(
    item_financials["abandoned"], 0.0, item_financials["effective_unit_profit"]
)
item_financials["expected_lost_profit"] = (
    item_financials["p_abandon"] * item_financials["effective_unit_profit"]
)

# Create totals view for headline impact
totals_view = (
    item_financials[
        [
            "potential_revenue",
            "realized_revenue",
            "expected_lost_revenue",
            "potential_profit",
            "realized_profit",
            "expected_lost_profit",
        ]
    ]
    .sum()
    .to_frame(name="total")
)

# Create product-mix view with revenue impact by category
dim = "beverage_category"  # change to any of the bucketed columns if desired
mix = (
    item_financials.groupby(dim, dropna=False)[
        [
            "potential_revenue",
            "realized_revenue",
            "expected_lost_revenue",
            "potential_profit",
            "realized_profit",
            "expected_lost_profit",
        ]
    ]
    .sum()
    .sort_values("expected_lost_revenue", ascending=False)
    .reset_index()
)

# Compute potential and realized mix shares and the shift
mix["potential_mix_share"] = (
    mix["potential_revenue"] / mix["potential_revenue"].sum()
)
mix["realized_mix_share"] = mix["realized_revenue"] / mix["realized_revenue"].sum()
mix["mix_shift"] = mix["realized_mix_share"] - mix["potential_mix_share"]

# Display headline totals and the top rows of mix
print(totals_view)
print(mix.head(10))

# Plot top categories by expected lost revenue
top_n = 10
_top = mix.head(top_n)
plt.figure(figsize=(10, 4))
plt.barh(_top[dim].astype(str)[::-1], _top["expected_lost_revenue"][::-1])
plt.xlabel("Expected Lost Revenue")
plt.title(f"Top {top_n} {dim} by Expected Lost Revenue")
plt.tight_layout()
plt.show()

# Plot potential vs realized vs expected lost revenue totals
totals_bar = pd.DataFrame(
    {
        "Potential": [mix["potential_revenue"].sum()],
        "Realized": [mix["realized_revenue"].sum()],
        "Expected Lost": [mix["expected_lost_revenue"].sum()],
    }
)
plt.figure(figsize=(6, 4))
plt.bar(totals_bar.columns, totals_bar.iloc[0].values)
plt.ylabel("Revenue")
plt.title("Potential vs Realized vs Expected Lost Revenue")
plt.tight_layout()
plt.show()

The above analysis shows that cart abandonment results in an estimated financial loss of approximately $4.9 million, which represents 19.4 percent of MyCoke360's total potential revenue of $25.3 million. Realized revenue totals $21.8 million, indicating that nearly one in every five dollars of potential sales is lost when customers fail to complete their purchases. The logistic regression model that produced these estimates achieved an AUC of 0.636, reflecting moderate predictive strength. This means that while the model provides a reasonable quantitative estimate of the scale of lost revenue, its precision is limited, and some drivers of abandonment may remain unaccounted for.

At the category level, Core Sparkling accounts for the largest share of lost revenue, estimated at over $6 million, followed by Energy Drinks at roughly $3 million and Packaged Water at just under $2 million. These three categories alone represent more than 75 percent of total expected lost revenue. Lower-volume segments such as Tea, Juices/Nectars, and Other Nonalcoholic Beverages show minimal financial loss in comparison.

Overall, the results quantify a substantial financial gap driven by cart abandonment, concentrated in MyCoke360's highest-demand product categories. Addressing abandonment behavior in these areas could potentially recover several million dollars annually and improve the balance of revenue contribution across the product mix.

In [0]:
# Gradient Boosted Trees (XGBoost) model for abandonment prediction
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist"
)

# Create pipeline with preprocessing and classifier
pipe_xgb = Pipeline([("pre", pre), ("clf", xgb)])

# Define hyperparameter grid for tuning
param_grid_xgb = {
    "clf__n_estimators": [150, 300],
    "clf__max_depth": [3, 6],
    "clf__learning_rate": [0.05, 0.1],
    "clf__subsample": [0.8, 1.0],
    "clf__colsample_bytree": [0.8, 1.0],
}

# Perform grid search with 3-fold cross-validation
cv_xgb = GridSearchCV(
    pipe_xgb,
    param_grid=param_grid_xgb,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)
cv_xgb.fit(X_train, y_train)

# Retrieve best model and parameters
print(f"Best XGB params: {cv_xgb.best_params_}")
best_xgb = cv_xgb.best_estimator_

# Evaluate model on test data
y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]
auc_xgb = roc_auc_score(y_test, y_prob_xgb)
print(f"XGB Test AUC: {auc_xgb:.3f}")

# Plot ROC curve for model performance
fpr, tpr, _ = roc_curve(y_test, y_prob_xgb)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"AUC={auc_xgb:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("XGBoost ROC")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.show()

The XGBoost ROC plot shows an AUC of 0.813, indicating strong predictive performance and a substantial improvement over the logistic regression model's AUC of 0.636. This means the XGBoost model can distinguish between completed and abandoned carts with a high degree of accuracy, correctly ranking outcomes more than 80 percent of the time.

In the context of understanding the financial impact of cart abandonment on MyCoke360, this improved performance means that any estimates or insights derived from the model would be significantly more reliable. If applied to the same revenue and product mix analysis, XGBoost would be expected to produce results that align directionally with earlier findings but with greater precision and confidence. The stronger discrimination ability indicates that this model can more accurately identify behavioral and transactional patterns contributing to revenue loss, making it a more efficient and dependable tool for quantifying and addressing the effects of cart abandonment.

In [0]:
# Score abandonment probability for each item
X_full = item_financials[
    [
        "effective_unit_price",
        "effective_unit_cost",
        "effective_unit_profit",
        "trademark",
        "flavor",
        "beverage_category",
        "packaging_type",
        "packaging_size",
    ]
].copy()

item_financials["p_abandon_xgb"] = best_xgb.predict_proba(X_full)[:, 1]

# Compute expected financial values
item_financials["expected_lost_revenue_xgb"] = (
    item_financials["p_abandon_xgb"] * item_financials["effective_unit_price"]
)
item_financials["expected_lost_profit_xgb"] = (
    item_financials["p_abandon_xgb"] * item_financials["effective_unit_profit"]
)

# Summarize total financial impact
totals_xgb = (
    item_financials[
        ["effective_unit_price", "expected_lost_revenue_xgb", "expected_lost_profit_xgb"]
    ]
    .sum()
    .rename(
        {
            "effective_unit_price": "Total Potential Revenue",
            "expected_lost_revenue_xgb": "Expected Lost Revenue",
            "expected_lost_profit_xgb": "Expected Lost Profit",
        }
    )
    .to_frame(name="total")
)

print(totals_xgb)

# Summarize expected loss by product category
impact_by_category = (
    item_financials.groupby("beverage_category", dropna=False)[
        ["expected_lost_revenue_xgb", "expected_lost_profit_xgb"]
    ]
    .sum()
    .sort_values("expected_lost_revenue_xgb", ascending=False)
    .reset_index()
)
print(impact_by_category.head(10))

# Visualize top categories by expected lost revenue
plt.figure(figsize=(10, 4))
plt.barh(
    impact_by_category["beverage_category"].astype(str)[::-1],
    impact_by_category["expected_lost_revenue_xgb"][::-1],
)
plt.xlabel("Expected Lost Revenue")
plt.title("Top Beverage Categories by Expected Lost Revenue (XGBoost Model)")
plt.tight_layout()
plt.show()

The XGBoost abandonment model provides a refined estimate of the financial exposure associated with cart abandonment on MyCoke360. Based on item-level abandonment probabilities and unit-level financials, the platform shows a total potential revenue of approximately 25.3 million dollars, an expected lost revenue of about 3.4 million dollars, and an expected lost profit of roughly 1.47 million dollars. These figures imply that approximately 13 to 14 percent of potential revenue is at risk due to abandonment behavior.

The model further breaks down financial impact by beverage category. The results indicate that losses are distributed across many product lines, with a smaller number of categories driving the largest share of expected lost revenue. According to the output, Core Sparkling represents the highest estimated loss at approximately 836,000 dollars, followed by Energy Drinks at around 123,000 dollars and Packaged Water (Plain & Enriched) at roughly 127,000 dollars. Additional categories such as Dairy/Soy Beverages, Enhanced Water Beverages, and Sports Drinks also contribute meaningfully, each generating expected losses in the 50,000 to 70,000 dollar range.

Although Core Sparkling is the clear outlier with the largest loss potential, the remaining categories show a much more gradual decline. Several smaller segments including Tea, Juices/Nectars, Fruit/Vegetable Still Drinks, and Other Nonalcoholic Beverages each contribute between 30,000 and 45,000 dollars in expected losses. This pattern suggests a long-tail distribution where many categories contribute modest but non-trivial losses.

Overall, the model's results confirm that cart abandonment continues to reduce realized revenue across a broad portion of the product catalog, with Core Sparkling exerting the greatest influence on total impact. Because the XGBoost model achieved a strong predictive performance with an AUC of 0.813, these estimates are considered reliable and provide a detailed, category-specific view of financial exposure. This insight highlights opportunities for targeted conversion improvements, particularly within Core Sparkling and the mid-tier beverage categories where losses remain substantial but potentially recoverable.

In [0]:
# Compute expected lost revenue and profit per item
probs_full = best_xgb.predict_proba(X)[:, 1]
item_activity_copy["p_abandon"] = probs_full
item_activity_copy["expected_lost_revenue"] = item_activity_copy["p_abandon"] * item_activity_copy["effective_unit_price"]
item_activity_copy["expected_lost_profit"] = item_activity_copy["p_abandon"] * item_activity_copy["effective_unit_profit"]

# Summarize actual and expected values
scenario_summary = pd.DataFrame(
    {
        "value": [
            item_activity_copy["realized_revenue"].sum(),
            item_activity_copy["potential_revenue"].sum(),
            item_activity_copy["abandoned_revenue"].sum(),
            item_activity_copy["expected_lost_revenue"].sum(),
            item_activity_copy["realized_profit"].sum(),
            item_activity_copy["potential_profit"].sum(),
            item_activity_copy["expected_lost_profit"].sum(),
        ]
    },
    index=[
        "actual_realized_revenue",
        "potential_revenue",
        "actual_abandoned_revenue",
        "expected_lost_revenue",
        "actual_realized_profit",
        "potential_profit",
        "expected_lost_profit",
    ],
)
display(scenario_summary)

# Simulate improvement scenario by reducing abandonment probability by 10%
item_activity_copy["p_abandon_improved"] = (item_activity_copy["p_abandon"] * 0.9).clip(0, 1)
item_activity_copy["exp_lost_rev_improved"] = (
    item_activity_copy["p_abandon_improved"] * item_activity_copy["effective_unit_price"]
)
item_activity_copy["exp_lost_profit_improved"] = (
    item_activity_copy["p_abandon_improved"] * item_activity_copy["effective_unit_profit"]
)

# Calculate revenue and profit recovered under improvement scenario
delta_rev = (
    item_activity_copy["expected_lost_revenue"].sum()
    - item_activity_copy["exp_lost_rev_improved"].sum()
)
delta_profit = (
    item_activity_copy["expected_lost_profit"].sum()
    - item_activity_copy["exp_lost_profit_improved"].sum()
)
print(f"Revenue recovered if abandonment falls 10%: ${delta_rev:,.0f}")
print(f"Profit recovered if abandonment falls 10%: ${delta_profit:,.0f}")

# Plot daily expected lost revenue: baseline vs improved scenario
daily_scn = (
    item_activity_copy.groupby("event_date", as_index=False)[
        ["expected_lost_revenue", "exp_lost_rev_improved"]
    ].sum()
)
plt.figure(figsize=(10, 4))
plt.plot(
    daily_scn["event_date"],
    daily_scn["expected_lost_revenue"],
    label="Baseline Expected Lost Rev",
)
plt.plot(
    daily_scn["event_date"],
    daily_scn["exp_lost_rev_improved"],
    label="Improved Scenario",
)
plt.title("Daily Expected Lost Revenue: Baseline vs Improved")
plt.xlabel("Date")
plt.ylabel("Expected Lost Revenue")
plt.legend()
plt.show()

The simulation results quantify the potential financial recovery if MyCoke360 reduces cart abandonment rates. Based on the model, a 10 percent reduction in cart abandonment over the period from June 2024 through May 2025 would recover approximately $339,538 in revenue and $142,743 in profit. This indicates that even modest improvements in conversion could generate measurable financial gains within a single year, reinforcing the importance of addressing abandonment behavior.

The time series comparison between baseline and improved scenarios shows that the effect of reduced abandonment would be consistent over time, with the improved scenario maintaining slightly lower expected lost revenue throughout the year. This pattern suggests that operational or marketing interventions aimed at improving conversion rates could deliver steady, cumulative revenue recovery rather than temporary spikes.

When considered alongside earlier findings, these results show that cart abandonment not only represents a significant unrealized revenue opportunity but also affects the overall product mix by limiting realized sales in high-volume categories such as Core Sparkling and Energy Drinks. Reducing abandonment would help shift the mix back toward these high-margin products, strengthening both total revenue and profitability across MyCoke360's beverage portfolio over the course of the year.

In [0]:
# Cluster items by abandonment and profit characteristics
item_mix = (
    item_activity_copy.groupby("item_id").agg(
        n_items=("item_id", "size"),
        abandon_rate=("abandoned", "mean"),
        mean_price=("effective_unit_price", "mean"),
        mean_profit=("effective_unit_profit", "mean"),
        category=(
            "beverage_category",
            lambda x: x.mode().iloc[0] if len(x.mode()) else "__unknown__",
        ),
    )
    .reset_index()
)

mix_features = item_mix[["abandon_rate", "mean_price", "mean_profit"]].fillna(0.0)

# Scale features for KMeans
scaler = StandardScaler()
X_mix = scaler.fit_transform(mix_features)

# Determine optimal number of clusters using elbow method
inertias = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_mix)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(range(2, 7), inertias, marker="o")
plt.title("Elbow Plot for Item Clusters")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.show()

The elbow plot was used to determine the optimal number of clusters for our model. The visual shows a clear bend at k = 3, indicating that three clusters provide the best balance between simplicity and explanatory power. Based on this result, our model uses k = 3 to group items by their abandonment, price, and profit characteristics for further analysis.

In [0]:
# Fit final KMeans model with k=3
km = KMeans(n_clusters=3, n_init=50, random_state=42)
item_mix["cluster"] = km.fit_predict(X_mix)

# Summarize clusters by product category
cluster_summary = (
    item_mix.groupby(["cluster", "category"])
    .agg(
        n_items=("item_id", "size"),
        avg_abandon_rate=("abandon_rate", "mean"),
        avg_profit=("mean_profit", "mean"),
    )
    .reset_index()
    .sort_values(["cluster", "n_items"], ascending=[True, False])
)

# Display top cluster summaries
display(cluster_summary.head(30))

# Compute and visualize cluster-level feature means
cluster_means = (
    item_mix.groupby("cluster")[["abandon_rate", "mean_price", "mean_profit"]]
    .mean()
    .reset_index()
)

cluster_means.plot(
    x="cluster",
    y=["abandon_rate", "mean_price", "mean_profit"],
    kind="bar",
    figsize=(8, 4),
)
plt.title("Cluster Means: Abandon Rate, Price, Profit")
plt.xlabel("Cluster")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

The clustering results reveal three distinct product groups that explain how cart abandonment affects MyCoke360's revenue and product mix in measurable ways.

Cluster 0 represents low-abandonment, high-profit products. It contains 39 items with an average abandonment rate of about 0.08 and an average profit near $44 per unit. This cluster includes categories such as Other Nonalcoholic Beverages, Juices/Nectars, and Fruit/Vegetable Still Drinks. While this group contributes modestly to total item volume, its high profitability means that each abandoned cart in this cluster carries significant financial weight.

Cluster 1 includes high-abandonment, low-profit items. It contains 60 items with an average abandonment rate of approximately 0.48 and an average profit around $12 per unit. This group is dominated by Energy Drinks, Core Sparkling, and Sports Drinks. Despite their strong sales potential, these categories experience the highest abandonment rates, suggesting they are major contributors to overall lost revenue. This cluster represents the largest area of opportunity for MyCoke360 to improve conversion rates and recover substantial revenue from high-traffic, lower-margin products.

Cluster 2 captures high-volume, stable performers. It contains over 600 items with an average abandonment rate of 0.09 and an average profit of about $10 per unit. The group is led by Core Sparkling and Energy Drinks, with supporting contributions from Sports Drinks and Packaged Water. Although abandonment rates are low, the scale of this cluster means it accounts for the majority of realized revenue. Small reductions in abandonment here would translate to significant absolute revenue recovery.

Quantitatively, the clustering shows that cart abandonment has both concentrated and distributed financial effects. The highest abandonment occurs in Cluster 1, where even small changes in customer behavior could recover large amounts of potential sales. Cluster 0's premium products magnify the profit impact of each lost sale, while Cluster 2's size means minor efficiency gains produce meaningful financial returns. Together, the results illustrate that abandonment reduces total revenue both by eroding high-margin sales and by limiting the conversion efficiency of MyCoke360's most popular product lines.

In [0]:
# Compute total revenue metrics
total_realized_rev = daily["realized_revenue"].sum()
total_potential_rev = daily["potential_revenue"].sum()
total_lost_rev = daily["abandoned_revenue"].sum()

# Compute total profit metrics
total_realized_profit = daily["realized_profit"].sum()
total_potential_profit = daily["potential_profit"].sum()
total_lost_profit = daily["abandoned_profit"].sum()

# Display summary KPIs for business interpretation
print(f"Realized Revenue: ${total_realized_rev:,.0f}")
print(f"Potential Revenue (w/o abandonment): ${total_potential_rev:,.0f}")
print(f"Lost Revenue due to abandonment: ${total_lost_rev:,.0f}")
print()
print(f"Realized Profit: ${total_realized_profit:,.0f}")
print(f"Potential Profit (w/o abandonment): ${total_potential_profit:,.0f}")
print(f"Lost Profit due to abandonment: ${total_lost_profit:,.0f}")

These results provide a clear quantitative baseline for understanding the financial impact of cart abandonment on MyCoke360's total revenue and product mix.

Total potential revenue without abandonment is approximately $25.3 million, while realized revenue is $21.8 million, meaning that about $3.46 million, or roughly 13.7 percent, of potential sales value is lost due to customers failing to complete their purchases. Similarly, potential profit is estimated at $10.47 million, but only $9.02 million is realized, resulting in an estimated $1.45 million in lost profit, also around 13.9 percent.

These figures indicate that cart abandonment represents a substantial financial constraint on platform performance. The scale of lost revenue and profit suggests that even small improvements in conversion rates could return hundreds of thousands of dollars annually. This aligns with later model-based findings showing that addressing abandonment behavior could recover significant value, especially within high-volume and high-margin product categories such as Core Sparkling and Energy Drinks.

Overall, this summary establishes that cart abandonment directly reduces both total revenue and profitability by more than ten percent, confirming it as a major driver of unrealized financial potential for MyCoke360.

## Q6 Modeling

In [0]:
# Aggregate behavioral features
customer_features = combined_with_customer.groupby('customer_id').agg(
    total_abandons=('is_abandon', 'sum'),
    total_purchases=('is_purchase', 'sum'),
    total_orders=('is_order', 'sum'),
    unique_devices=('device_category', 'nunique'),
    unique_mobile_brands=('device_mobile_brand_name', 'nunique'),
    unique_os=('device_operating_system', 'nunique'),
    avg_order_quantity=('order_quantity', 'mean')
).reset_index()

static_cols = [
    'customer_id', 'sales_office_location', 'cold_drink_channel_description',
    'customer_sub_trade_channel_description', 'distribution_mode',
    'shipping_duration', 'shipping_destination'
]

customer_features = customer_features.merge(
    combined_with_customer[static_cols].drop_duplicates('customer_id'),
    on='customer_id', how='left'
)

# adding target variable
customer_features = customer_features.merge(
    combined_with_customer[['customer_id', 'churned']].drop_duplicates('customer_id'),
    on='customer_id', how='left'
)
customer_features['churned'] = customer_features['churned'].astype(int)

# Encode categorical variables
categorical_cols = [
    "sales_office_location",
    "cold_drink_channel_description",
    "customer_sub_trade_channel_description",
    "distribution_mode",
    "shipping_duration",
    "shipping_destination",
]

for col in categorical_cols:
    le = LabelEncoder()
    customer_features[col] = le.fit_transform(customer_features[col].astype(str))

# Define features and target
X = customer_features.drop(["customer_id", "churned"], axis=1)
y = customer_features["churned"]

# Impute missing numeric values
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

# Initialize random forest model
rf = RandomForestClassifier(n_estimators=200, random_state=42)

# Create stratified 5-fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize metric trackers and importance accumulator
accuracy_scores, recall_scores, precision_scores, f1_scores, auc_scores = [], [], [], [], []
feature_importances = np.zeros(X.shape[1])

# Perform cross-validation loop
for train_idx, test_idx in skf.split(X_imputed, y):
    X_train, X_test = X_imputed[train_idx], X_imputed[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    y_prob = rf.predict_proba(X_test)[:, 1]

    # Collect performance metrics
    accuracy_scores.append(accuracy_score(y_test, y_pred))
    recall_scores.append(recall_score(y_test, y_pred))
    precision_scores.append(precision_score(y_test, y_pred))
    f1_scores.append(f1_score(y_test, y_pred))
    auc_scores.append(roc_auc_score(y_test, y_prob))

    # Accumulate feature importances
    feature_importances += rf.feature_importances_

# Compute average feature importances
feature_importances /= skf.get_n_splits()

# Create feature importance dataframe
feat_imp_df = pd.DataFrame({
    "feature": X.columns,
    "importance": feature_importances
}).sort_values(by="importance", ascending=False)

# Print summary results
print("---- Cross-Validation Results ----")
print(f"Mean Accuracy:  {np.mean(accuracy_scores):.3f}")
print(f"Mean Recall:    {np.mean(recall_scores):.3f}")
print(f"Mean Precision: {np.mean(precision_scores):.3f}")
print(f"Mean F1 Score:  {np.mean(f1_scores):.3f}")
print(f"Mean ROC-AUC:   {np.mean(auc_scores):.3f}")

feat_imp_df["importance"] = feat_imp_df["importance"].map(lambda x: f"{x:.6f}")

Using a Random Forest model evaluated with 5-Fold Cross Validation, we achieved a mean accuracy of 0.905, recall of 0.604, and precision of 0.739 in predicting customer churn. This means the model correctly identifies churners approximately 74% of the time, but it misses about 40% of actual churners. While the overall accuracy and precision are strong, the moderate recall indicates that the model potentially prioritizes precision over sensitivity. The model also achieved a high ROC-AUC of 0.951, suggesting excellent ability to distinguish between churners and non-churners across thresholds.

In [0]:
# Prepare ranked feature-importance table
fi = feat_imp_df.copy()
fi = fi[["feature", "importance"]].dropna()
fi["feature"] = fi["feature"].astype(str)
fi["importance"] = pd.to_numeric(fi["importance"], errors="coerce")
fi = fi.dropna(subset=["importance"])

# Sort by importance and add a 1-based rank index
fi_sorted = fi.sort_values("importance", ascending=False).reset_index(drop=True)
fi_sorted.index = fi_sorted.index + 1
fi_table = fi_sorted.rename_axis("rank").reset_index()

# Display table (Databricks will render this like the image)
fi_table

# Plot feature importance to match the table order
plt.figure(figsize=(10, 6))
plt.barh(fi_sorted["feature"], fi_sorted["importance"], color="skyblue")
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

The most important predictor in churn was site interactions and order volume. Clients who ordered more generally had a lower likelihood of churning. Abandonment is also a good indicator, with an importance weight of roughly 0.13, indicating that clients who abandon carts are more likely to churn than those who dont. Other important indicators include sub_trade, cold_drink_channel, and sales_office_location. We will take a closer look at these metrics in the coming graphs.

In [0]:
# Summarize churn by sales office location
churn_summary = (
    combined_with_customer
    .groupby("sales_office_location")
    .agg(
        total_customers_who_abandoned=("customer_id", lambda x: x.nunique()),
        churned_customers=("customer_id", lambda x: x[combined_with_customer.loc[x.index, "churned"]].nunique()),
    )
    .reset_index()
)

# Calculate churn rate percentage
churn_summary["churn_rate"] = (
    churn_summary["churned_customers"] / churn_summary["total_customers_who_abandoned"]
) * 100

# Sort results by churn rate
churn_summary_sorted = churn_summary.sort_values("churn_rate", ascending=False)

# Plot churn rate by location
plt.figure(figsize=(10, 6))
plt.barh(
    churn_summary_sorted["sales_office_location"],
    churn_summary_sorted["churn_rate"],
    color="skyblue",
)
plt.xlabel("Churn Rate (%)")
plt.ylabel("Sales Office Location")
plt.title("Churn Rate Among Customers Who Abandoned, by Sales Office")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

There are three sales offices that had the highest churn rate among customers who abandoned a cart: Glenwood Springs, CO, Alamosa, CO, and Cheyenne, WY. These each had a churn rate of roughly 28%. The two sales offices with the lowest churn rate of zero were Pedleton, OR, and Elko, NV.

In [0]:
# Summarize churn metrics by sub trade channel
churn_summary = (
    combined_with_customer
    .groupby("customer_sub_trade_channel_description")
    .agg(
        total_customers_who_abandoned=("customer_id", lambda x: x.nunique()),
        churned_customers=("customer_id", lambda x: x[combined_with_customer.loc[x.index, "churned"]].nunique()),
    )
    .reset_index()
)

# Calculate churn rate percentage
churn_summary["churn_rate"] = (
    churn_summary["churned_customers"] / churn_summary["total_customers_who_abandoned"]
) * 100

# Sort summary by churn rate
churn_summary_sorted = churn_summary.sort_values("churn_rate", ascending=False)

# Plot churn rates by sub trade channel
plt.figure(figsize=(10, 6))
plt.barh(
    churn_summary_sorted["customer_sub_trade_channel_description"],
    churn_summary_sorted["churn_rate"],
    color="skyblue",
)
plt.xlabel("Churn Rate (%)")
plt.ylabel("customer_sub_trade_channel_description")
plt.title("Churn Rate Among Customers Who Abandoned, by Sub Trade Channel")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

First thing to note is that our graph only goes up to 50%, so among specific sub-trade channels, none had a 100% churn rate for customers who abanoned a cart. However, mini-markets, military facilities, and general retail all had a 50% churn rate. Food trucks also had a high churn rate, along with airlines, religious institutions, and amusement centers. Sub-trades with the lowest churn rates include warehouse clubs, schools, theme parks, convenience stores and pharmacies. 

In [0]:
# Sankey diagram of channel switches

# Ensure correct dtypes
ss = switch_summary.copy()
ss["from_channel"] = ss["from_channel"].astype(str)
ss["to_channel"] = ss["to_channel"].astype(str)
ss["count"] = ss["count"].astype(int)

# Build node list and index maps
labels = pd.Index(pd.unique(ss[["from_channel", "to_channel"]].values.ravel("K"))).tolist()
label_to_idx = {lbl: i for i, lbl in enumerate(labels)}

# Map to source/target arrays
sources = ss["from_channel"].map(label_to_idx).tolist()
targets = ss["to_channel"].map(label_to_idx).tolist()
values = ss["count"].tolist()

# Create Sankey
fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(label=labels, pad=15, thickness=18),
            link=dict(source=sources, target=targets, value=values),
        )
    ]
)
fig.update_layout(title_text="Post Abandonment Channel Switch Flows", font_size=12, height=500)
fig.show()

The largest ordering shift, by far, came from clients who moved from a sales_rep to mycoke360, with nearly 300 instances of switching. The next two highest switches were also to mycoke360, coming from mycoke_legacy and the call center respectively. So despite abandoning a cart, there was still a large majority of clients who chose to switch their ordering method to mycoke360. The 'other' category had the lowest number of clients switching into it post-abandonment. Of the identifiable methods, the least amount of people switch to using sales_reps.

In [0]:
# Heatmap of from_channel vs to_channel

# Build matrix
ss = switch_summary.copy()
ss["from_channel"] = ss["from_channel"].astype(str)
ss["to_channel"] = ss["to_channel"].astype(str)
ss["count"] = ss["count"].astype(int)

mat = ss.pivot_table(
    index="from_channel",
    columns="to_channel",
    values="count",
    aggfunc="sum",
    fill_value=0,
).astype(int)

# Plot heatmap
plt.figure(figsize=(8, 6))
im = plt.imshow(mat.values, aspect="auto")
plt.colorbar(im, label="Count")

# Axis labels and ticks
plt.xticks(ticks=np.arange(mat.shape[1]), labels=mat.columns, rotation=45, ha="right")
plt.yticks(ticks=np.arange(mat.shape[0]), labels=mat.index)
plt.xlabel("to_channel")
plt.ylabel("from_channel")
plt.title("Post Abandonment Channel Switch Counts Heatmap")

# Optional annotations
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        val = mat.iat[i, j]
        if val > 0:
            plt.text(j, i, str(val), ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()

This visualizes the Sankey graph in another way. The largest ordering shift, by far, came from clients who moved from a sales_rep to mycoke360, with nearly 300 instances of switching. The next two highest switches were also to mycoke360, coming from mycoke_legacy and the call center respectively. So despite abandoning a cart, there was still a large majority of clients who chose to switch their ordering method to mycoke360. The 'other' category had the lowest number of clients switching into it post-abandonment. Of the identifiable methods, the least amount of people switch to using sales_reps.

In [0]:
# Filter switches directed to mycoke360
to_mycoke360 = (
    switch_df[switch_df["to_channel"] == "mycoke360"]["to_channel"]
    .value_counts()
    .reset_index()
)
to_mycoke360.columns = ["channel", "count"]
to_mycoke360["direction"] = "to_mycoke360"

# Filter switches originating from mycoke360
from_mycoke360 = (
    switch_df[switch_df["from_channel"] == "mycoke360"]["from_channel"]
    .value_counts()
    .reset_index()
)
from_mycoke360.columns = ["channel", "count"]
from_mycoke360["direction"] = "from_mycoke360"

# Combine switch data
switch_plot_df = pd.concat([to_mycoke360, from_mycoke360], ignore_index=True)

# Visualize switch counts
plt.figure(figsize=(8, 6))
plt.bar(
    switch_plot_df["direction"],
    switch_plot_df["count"],
    color=["skyblue", "salmon"]
)
plt.ylabel("Number of Switches")
plt.xticks(rotation=45, ha="right")
plt.title("Switches To and From MyCoke360")
plt.tight_layout()
plt.show()

Overall, there was a much greater shift of clients going into mycoke360 than there were leaving post abandonment. This indicates that abandonment may not be indicative of a poor user experience.

# Results

## Q1 Behavioral Events & Sequences

What behavioral events or sequence of events are the strongest predictors of cart abandonment?

There are clear event_names that correspond to cart abandonment. In reviewing patterns/sequences leading up to abandonment, the uniqueness of each sequence inhibited statistical significance of the findings. Roughly 75% of abandoned carts had the previously identified 10 event_names as the last event in the order window sequence. In order to leverage this information, we mined the average amount of time from when these events were clicked, and when a normal non-recovery purchase was made. From these insights, we found the following events, on average, have a 75% chance of a purchase taking place within one day of being executed. These events encompass 70% of abandoned carts

1. update_cart
2. remove_from_cart
3. view_item_list
4. page_view - Unknown Page
5. page_view - Mycoke Dashboard       
6. button_click - Mycoke Orders - Cart
7. page_view - Mycoke Orders
8. button_click - Mycoke Orders - Checkout: Review Order
9. proceed_to_checkout
 
The following events on average have a 85% chance of a purchase taking place within one day of being executed. These events encompass 46% of abandoned cart segments:

1. update_cart
2. remove_from_cart
3. button_click - Mycoke Orders - Cart
4. button_click - Mycoke Orders - Checkout: Review Order
5. proceed_to_checkout
 
From these results we can make clear recommendations as to when Swire should reach out to customers based on their user behavior. They have the option to be conservative and reach out on events in the 85% threshold or try to reduce cart abandonment by a larger threshold and reach out on events in the 75% threshold.

## Q2 Behaviors & Conditions

What behaviors or conditions lead to a customer returning to complete a previously abandoned cart?

The current recovery rate is roughly 53%. When looking at cart recovery, 50% of total carts are recovered in 2 hours, with only an additional 15%  (65% total) occurring in 3 days. Therefore, there is an incredibly steep decline in recovery rate over time. We can glean two major insights from these results. The first is that customer recovery tends to be intentional. If they meant to check out a cart, they will very shortly after their order window. Otherwise, they'll let it sit. The second is that time from abandonment is a good indicator of whether recovery will occur. It can be assumed that if a cart is not recovered within 1 or 2 days, it was intentional and will be abanoned. This supports the Q1 recommendation that sending reminders within 1 day of the identified event_names will target customers who actually meant to purchase, but for some reason did not.

## Q3 Variance by Device Type

How does cart abandonment vary by device type and how can it be reduced?

Out of all device types, Tablets have the highest abandoned rates followed by mobile, then desktops. With an abandoned rate of 21.8% for tablets, this could be due to UI difficulties. This is significant as tablets themselves are the rarest device used when navigating through MyCoke360, yet represent the highest abandon rate. Mobile is close to the abandonment rate as Tablet at 20.7%. 

## Q4 Product Frequency

Which products appear most frequently in abandoned carts?

Based on our aggregated analysis of all abandoned cart events, Fizz Factory is the most frequently abandoned brand on the platform, appearing in approximately 250,000 abandoned carts. This represents a significantly larger share of abandonment volume compared to other brands, suggesting either high browsing interest without conversion or potential friction related to those products. The second most abandoned brand is Oliver Originals, with roughly 145,000 abandoned carts, followed by Pete's Popcorn at approximately 60,000. 

## Q5 Financial Impact

What is the financial impact of cart abandonment on total MyCoke360 revenue and product mix?

Cart abandonment has a clear and measurable financial impact on MyCoke360's revenue and product mix. Across the analysis, total potential revenue is estimated at $25.3 million, with $21.8 million realized, resulting in approximately $3.5 million in lost revenue and $1.45 million in lost profit, or about 14 percent of total potential sales. Both the SARIMA and Prophet models confirm that this gap between potential and realized revenue is consistent over time, with losses increasing alongside overall platform growth.

The XGBoost model, which achieved an AUC of 0.813, provides a reliable estimate of these losses and identifies that most of the financial impact is concentrated in a few key categories. Core Sparkling, Energy Drinks, and Packaged Water account for roughly 70 percent of total lost revenue, showing that abandonment is most pronounced among high-demand, high-volume products. Clustering analysis further supports this finding, revealing three product groups that vary in both profitability and abandonment behavior. High-profit items experience the greatest proportional loss, while large-volume, moderate-margin products account for the largest total revenue effect.

Simulation results show that even modest improvements could yield meaningful gains. A 10 percent reduction in cart abandonment over a one-year period (June 2024–May 2025) would recover an estimated $339,538 in revenue and $142,743 in profit, with benefits distributed steadily throughout the year.

Taken together, these findings demonstrate that cart abandonment reduces MyCoke360's total realized revenue by over ten percent and distorts the product mix by limiting conversions in its most profitable and frequently purchased beverage categories. Addressing this issue through checkout optimization, targeted engagement, and category-specific interventions could restore millions in annual revenue while improving the overall profitability and balance of the product portfolio.

## Q6 Post-Abandonment

What happens after a cart is abandoned? Does the customer order through another method or churn?

Roughly 86.5% of customer remain active after their first instance of abandonment with only about 13.5% churning. This indicates that abandonment alone is not a strong indicator of customer loss. Instead, many customers continue to engage and even order more often through mycoke360.

A significant portion of customers switch ordering channels after abandoning. The most common transition by far was from Sales Rep to myCoke360, with nearly 300 instances recorded. Other frequent switches also led to myCoke360, coming from the myCoke_Legacy and Call Center channels. Very few customers moved away from myCoke360, suggesting that abandonment often signals a shift toward digital self-service ordering rather than disengagement or dissatisfaction.

Churn rates after abandonment varied across regions and customer types. The highest churn rates were observed among customers associated with sales offices in Glenwood Springs, CO; Alamosa, CO; and Cheyenne, WY, each at about 28%. Among sub-trade channels, mini-markets, military facilities, and general retail had the highest churn rates of roughly 50%, followed by food trucks, airlines, religious institutions, and amusement centers. In contrast, warehouse clubs, schools, convenience stores, and pharmacies exhibited the lowest churn rates.

A Random Forest model trained to predict churn achieved strong results, with a mean accuracy of 0.905, recall of 0.604, precision of 0.739, and a ROC-AUC of 0.951. The most important predictors of churn included site interactions, order volume, and cart abandonment. Customers who ordered more frequently or interacted more with the site were less likely to churn, while those with a history of abandonment were more likely to leave.

## Recommendations

### Timely Cart Reminders

We recommend Coca Cola Swire reach out to customers within 1 day of the 75% threshold events, if there are items in their cart, to remind them they have yet to checkout. On average, if a purchase is to occur, there is a 75% it will follow within one day from the event_names listed in the Q1 results. Also, if Swire reaches out only when items are in the cart, it will reduce the amount of times customers receive the reminder, making it more targeted. With this recommendation Swire can potentially reduce cart abandonment by 70% if this process is successful. As discussed in the Q2 results later on, recovered carts are typically recovered within 2 hours, with a plateau in recovery occurring after the 2 hour threshold. If a customer genuinely does not mean to purchase, then they will ignore Swire's reminder. However, if something happened where the customer actually meant to check out the cart, but did not, the reminder will bring them back to checkout. 

### Benchmark Recovery by Device Type

I would recommend Swire benchmark the recovery rate by device type when assessing how successful the recommendation for Q1 is. This will contextualize the effect of the cart reminders by device type, to understand if reminders are more effective for some users than others. For example, Mozilla mobile users have a roughly 20% higher recovery rate than Apple mobile users within the first hour of abandonment. If the reminder emails are effective, then both rates will increase. However, if Apple recovery within 1 hour increases to be equal with Mozilla, then we know the tactic is effective for Apple users, but not Mozilla users. 

### Optimize Mobile and Tablet Experience

The abandonment rate in tablets and mobile could indiciate difficulties due to ordering with a smaller screen size. Revising the MyCoke360 webpage to ease Tablet and Mobile orders could be worth implementing and the results can be later compared via an A/B test.

### Target High-Abandonment Brands

The sharp drop off after the first two brands (Fizz Factory and Oliver Originals) indicates a distribution where only a small number of brands account for the majority of abandonment. These insights can help prioritize retention strategies, such as targeted promotions, improved product page content, or restocking visibility for the most affected brands which SWIRE can implement.

### Reduce Financial Losses from Cart Abandonment

Cart abandonment analysis shows that MyCoke360 is losing roughly 13 to 14 percent of potential revenue each year, with the majority of this loss coming from high-volume categories such as Core Sparkling and Energy Drinks. Addressing this behavior requires a combination of process improvements, targeted engagement, and data-driven intervention.

Reducing friction in the checkout process should be a primary focus. Simplifying payment steps, improving page load times, and enabling saved preferences for frequent customers would directly address one of the most common causes of abandonment. Because the SARIMA and Prophet models show that abandonment peaks during high-traffic periods, these optimizations would not only improve conversion rates but also have a compounding effect during seasonal or promotional spikes when the platform experiences its largest revenue gaps.

Predictive insights from the XGBoost model can be used to identify at-risk customers and trigger timely recovery actions such as reminder notifications, retargeting campaigns, or personalized discounts. These measures would be especially valuable for high-margin, frequently abandoned products, where the financial payoff per recovered sale is largest. Similarly, targeted pricing or bundling strategies could be applied to lower-margin, high-abandonment categories to increase conversion without reducing overall profitability.

Clustering results suggest that product-level strategies should be differentiated. High-profit, low-abandonment products benefit most from maintaining premium positioning and convenience-focused incentives, while high-abandonment, low-profit items require value reinforcement through bundling, discounts, or subscription options.

Given that even a 10 percent reduction in abandonment over a single year could recover approximately $340,000 in revenue and $143,000 in profit, the company should prioritize developing a continuous monitoring framework. Tracking realized versus potential revenue in real time will allow MyCoke360 to measure progress, evaluate intervention success, and refine customer engagement strategies dynamically.

Taken together, these steps would allow MyCoke360 to recover significant lost revenue, strengthen profitability across key beverage categories, and restore balance in its product mix by converting more high-value purchase opportunities into completed sales.

### Post-Abandonment Retention Strategy

Overall, after abandonment, most customers stay active and many shift their purchasing behavior toward myCoke360. Rather than viewing abandonment as a loss event, it appears to be part of a broader behavioral transition. Therefore, post-abandonment strategies should focus on supporting and smoothing these channel transitions, particularly for customers moving to digital self-service. In addition, monitoring high-churn regions and sub-trade channels can help identify where targeted retention efforts will have the greatest impact.
